# SISALv3 Speleothem Geochemical Imputation — Clean Publication Pipeline

**Journal:** *Applied Computing and Geosciences* (under revision)
**Manuscript:** *Evaluating Machine Learning for Gap Filling in Speleothem Geochemical Records*

This pipeline is the single source of truth for every number, table, and figure in the manuscript. It supersedes the seven exploratory notebooks (`Copy of SISALv3_Speleothem_RandomForest (1).ipynb`, `…_XGBoost.ipynb`, `…_Transformer.ipynb`, `SISALv3_Speleothem_LSTM_Bidirectional.ipynb`, `SISALv3_LeaveOneCaveOut_Validation.ipynb`, `SISALv3_CubicSpline_Baseline_Comparison.ipynb`, `SISALv3_Baseline_Cubic_Spline_Comparison.ipynb`), which contained methodological errors documented in the project Gap Analysis (data leakage in preprocessing, random-row splits ignoring cave grouping, sliding windows that crossed cave boundaries, and an index-based cubic spline that was effectively interpolating between physically-adjacent rows of the same cave).

## What this pipeline does differently

1. **Rebuilds the master table from raw SISALv3 SQL/CSVs** rather than from the pre-leaked `cleaned_data.csv`.
2. **Cave-aware cross-validation.** `LeaveOneGroupOut` over `site_id` is the headline (Tables 1–3); a `BlockedWithinCaveSplitter` powers the secondary within-cave gap-fill experiment (Section 3.5 / Table 4) where cubic spline becomes a meaningful baseline.
3. **All preprocessing lives inside `Pipeline`/`ColumnTransformer`** so imputation, scaling, and one-hot encoding are fit only on the training fold of each split.
4. **`OneHotEncoder` for nominal categoricals** (`mineralogy`, `monitoring`, `hemisphere`, `lat_band`) instead of `LabelEncoder`, with `handle_unknown="infrequent_if_exist"` so previously unseen levels in held-out caves do not crash inference.
5. **Sequence DL models (BiLSTM, Transformer) are built strictly per `entity_id`,** rows sorted by `interp_age` (with `depth_sample` as the chronological fallback). Sequences never cross entity boundaries.
6. **Cubic spline / linear interpolation** are evaluated **only** in the within-cave block-gap regime, where they are mathematically defined; under LOCO they are reported as *not applicable* with a stated rationale rather than a fake 0 % bar.
7. **Real permutation tests** (refit the entire pipeline on shuffled `y_train`, repeat M times) replace the previous broken implementation that only shuffled `y_test`.
8. **Reproducible artifact bundle** under `artifacts/<UTC-timestamp>/` so the manuscript and the code can never drift apart.

## How the pipeline is split into two notebooks

To stay within Google Colab Pro's per-session wall-time cap (~24 h, often less), the pipeline is delivered as **two notebooks** that share a Drive-resident artifact bundle:

| Notebook | Purpose | Default wall-time on Colab Pro | Writes |
|---|---|---|---|
| `001_SISALv3_Pipeline_PART1_Train_ENRICHED.ipynb` | Build master table (+ enriched covariates), hyper-parameter search, **LOCO loop** | ~5 h (tabular only); ~24 h with DL panel enabled | `master_table.parquet`, `best_params/`, `predictions/`, `metrics/loco_*` |
| `001_SISALv3_Pipeline_PART2_Evaluate.ipynb` | Within-cave gap-fill, paired-Wilcoxon, optional perm tests, **all plots & tables** | ~3–4 h (defaults); ~30+ h if perm tests enabled | `metrics/within_*`, `figures/`, `TABLE_*.csv`, `MANIFEST.json` |

PART 1 stamps a UTC timestamp into `artifacts/LATEST_RUN.txt`; PART 2 reads that file automatically, so you don't have to copy-paste the run id between sessions. Every heavy loop in both notebooks is **cached on disk per `(model, target)` pair**, so a Colab disconnect mid-run is recoverable simply by re-executing all cells of the same notebook.

---

> **You are reading PART 2 of 2 — the *evaluation & visualisation* notebook.** PART 1 (`001_SISALv3_Pipeline_PART1_Train_ENRICHED.ipynb`) must have finished at least once before this notebook is run; PART 2 reads the run id PART 1 wrote into `artifacts/LATEST_RUN.txt`. **CONFIG must match PART 1 ENRICHED** (WorldClim bioclims, Bowen–Wilkinson precip δ18O, `hemisphere`, `lat_band`). Default wall-time on Colab Pro is ~3–4 h; the optional permutation-test cell (section 4.5) is gated off by default because it can take 30+ h on its own.

---

## 0. Colab setup, dependencies, configuration & imports

**Runtime target:** Google Colab Pro (GPU runtime recommended for the BiLSTM / Transformer sections, but the whole pipeline also runs on CPU).

**Folder layout on Drive** (identical to the legacy notebooks — no folder restructuring required):

```
/content/drive/MyDrive/SISALv3_Speleothem/
├── data/
│   ├── cleaned_data.csv                       # legacy file — NOT used by this notebook
│   └── sisalv3_database_mysql_csv/
│       └── sisalv3_database_mysql_csv/
│           └── sisalv3_csv/                   # raw SISALv3 tables, used here
│               ├── site.csv, entity.csv, sample.csv, original_chronology.csv
│               ├── d18O.csv, d13C.csv, Mg_Ca.csv, Sr_Ca.csv
│               ├── P_Ca.csv, U_Ca.csv, Ba_Ca.csv, Sr_isotopes.csv
│               └── hiatus.csv, gap.csv
├── models/                                    # fitted models written here (1 per LOCO fold)
└── artifacts/<UTC-timestamp>/                 # per-run outputs (predictions, metrics, figs)
```

The six steps below are: (0.1) runtime toggle + Drive mount policy, (0.2) install dependencies that Colab does not pre-ship, (0.3) import everything, (0.4) resolve project paths, (0.5) define the single `CONFIG` object, (0.6) seed every RNG.

**Runtime toggle:** the first code cell exposes a single switch:

- `USER_RUNTIME_MODE = "auto"` (recommended) — chooses Colab when running on Colab, local otherwise.
- `USER_RUNTIME_MODE = "colab"` — force Colab/Drive mode (fails fast if not in Colab).
- `USER_RUNTIME_MODE = "local"` — force local mode (never mounts Drive).

Path compatibility intentionally matches the legacy notebooks: both
`/content/drive/MyDrive/SISALv3_Speleothem` and
`/content/drive/My Drive/SISALv3_Speleothem` are supported.

In [1]:
# --- 0.1 Runtime mode toggle (auto / colab / local) --------------------------------
# Single user-facing switch for every notebook variant.
#
#   "auto"  : use Colab mode when running in Colab, else local mode.
#   "colab" : force Colab mode (requires google.colab).
#   "local" : force local mode (never mounts Google Drive).
USER_RUNTIME_MODE = "auto"

_valid_modes = {"auto", "colab", "local"}
if USER_RUNTIME_MODE not in _valid_modes:
    raise ValueError(
        f"USER_RUNTIME_MODE={USER_RUNTIME_MODE!r} is invalid; "
        f"use one of {_valid_modes}."
    )

try:
    import google.colab  # noqa: F401
    _HAS_COLAB = True
except Exception:
    _HAS_COLAB = False

if USER_RUNTIME_MODE == "auto":
    RUNTIME_MODE = "colab" if _HAS_COLAB else "local"
else:
    RUNTIME_MODE = USER_RUNTIME_MODE

if RUNTIME_MODE == "colab" and not _HAS_COLAB:
    raise RuntimeError(
        "RUNTIME_MODE is 'colab' but google.colab is unavailable. "
        "Set USER_RUNTIME_MODE='local' or run in Colab."
    )

IN_COLAB = bool(RUNTIME_MODE == "colab")
LOCAL_PROFILE = bool(RUNTIME_MODE == "local")
print(
    f"[OK] Runtime mode resolved: {RUNTIME_MODE} "
    f"(USER_RUNTIME_MODE={USER_RUNTIME_MODE!r}, has_colab={_HAS_COLAB})"
)

[OK] Runtime mode resolved: colab (USER_RUNTIME_MODE='auto', has_colab=True)


In [2]:
# --- 0.1b Drive mount policy --------------------------------------------------------
if IN_COLAB:
    import os
    from google.colab import drive
    if os.path.isdir("/content/drive/MyDrive") or os.path.isdir("/content/drive/My Drive"):
        print("[OK] Google Drive already available at /content/drive")
    else:
        drive.mount("/content/drive", force_remount=False)
        print("[OK] Google Drive mounted at /content/drive")
else:
    print("[INFO] Local mode selected — Google Drive mount skipped.")

Mounted at /content/drive
[OK] Google Drive mounted at /content/drive


In [3]:
# --- 0.2 Install dependencies that Colab does not pre-ship ------------------------
#
# Colab Pro already provides:    numpy, pandas, scikit-learn, tensorflow, matplotlib,
#                                seaborn, scipy, joblib.
# We install / upgrade only what is genuinely missing or stale:
#   - scikeras    : sklearn-compatible KerasRegressor wrapper used by the BiLSTM /
#                   Transformer SequenceWrapper.
#   - xgboost     : pin to a recent 2.x for the categorical-aware tree splitter.
#   - catboost    : 5th model in the panel — ordered boosting + native categorical
#                   handling, complementary inductive bias to RF/XGB, particularly
#                   strong on LOCO (out-of-distribution) generalization.
#   - pyarrow     : Parquet I/O for the master table and per-fold prediction logs.
#   - cartopy/contextily not used — basemaps are hand-drawn from coastlines to keep
#                   Colab installs lightweight; matplotlib + seaborn only.
#
# We intentionally use subprocess+pipspec checks (not `%pip`) so this cell runs
# in Colab, local Jupyter, and headless nbconvert environments.
import importlib.util
import subprocess
import sys

REQUIRED = {
    "scikeras": "scikeras>=0.13",
    "xgboost": "xgboost>=2.0",
    "catboost": "catboost>=1.2",
    "pyarrow": "pyarrow>=14.0",
}
missing = [spec for mod, spec in REQUIRED.items() if importlib.util.find_spec(mod) is None]

if missing:
    print("[INFO] Installing missing packages:", ", ".join(missing))
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", "--upgrade", *missing])
    print("[OK] Missing dependencies installed.")
else:
    print("[OK] All required dependencies already installed.")

[INFO] Installing missing packages: scikeras>=0.13, catboost>=1.2
[OK] Missing dependencies installed.


In [4]:
# --- 0.3 Imports ------------------------------------------------------------------
from __future__ import annotations

import json
import logging
import os
import random
import sys
import warnings
from dataclasses import dataclass, field, replace
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Callable

import numpy as np
import pandas as pd

# Silence non-actionable warnings; never blanket-silence errors.
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning, module="sklearn")

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)-7s | %(message)s",
    datefmt="%H:%M:%S",
    stream=sys.stdout,
)
log = logging.getLogger("sisalv3")

log.info("Python %s | NumPy %s | Pandas %s",
         sys.version.split()[0], np.__version__, pd.__version__)

In [5]:
# --- 0.4 Project paths (legacy-compatible Colab + local) ---------------------------
#
# Legacy notebooks used both:
#   - /content/drive/MyDrive/SISALv3_Speleothem
#   - /content/drive/My Drive/SISALv3_Speleothem
# We support both and auto-pick the first existing one in Colab mode.
#
# Override with:
#   export SISAL_PROJECT_ROOT="/abs/path/to/SISALv3_Speleothem"
COLAB_ROOT_CANDIDATES = [
    Path("/content/drive/MyDrive/SISALv3_Speleothem"),
    Path("/content/drive/My Drive/SISALv3_Speleothem"),
]
DEFAULT_LOCAL_ROOT = Path.cwd() / "SISALv3_Speleothem"

if globals().get("IN_COLAB", False):
    existing = [p for p in COLAB_ROOT_CANDIDATES if p.exists()]
    DEFAULT_PROJECT_ROOT = existing[0] if existing else COLAB_ROOT_CANDIDATES[0]
else:
    DEFAULT_PROJECT_ROOT = DEFAULT_LOCAL_ROOT

PROJECT_ROOT = Path(os.environ.get("SISAL_PROJECT_ROOT", str(DEFAULT_PROJECT_ROOT))).resolve()

RAW_SISAL_DIR = (
    PROJECT_ROOT
    / "data"
    / "sisalv3_database_mysql_csv"
    / "sisalv3_database_mysql_csv"
    / "sisalv3_csv"
)
MODELS_DIR = PROJECT_ROOT / "models"
ARTIFACTS_ROOT = PROJECT_ROOT / "artifacts"
DATA_OUT_DIR = ARTIFACTS_ROOT / "_data"   # cohort-wide cache shared across runs
LATEST_RUN_FILE = ARTIFACTS_ROOT / "LATEST_RUN.txt"

ARTIFACTS_ROOT.mkdir(parents=True, exist_ok=True)
DATA_OUT_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)

log.info("PROJECT_ROOT     : %s", PROJECT_ROOT)
log.info("RAW_SISAL_DIR    : %s", RAW_SISAL_DIR)
log.info("Models dir       : %s", MODELS_DIR)
log.info("Persistent cache : %s", DATA_OUT_DIR)
log.info("Artifacts root   : %s", ARTIFACTS_ROOT)

In [6]:
# --- 0.4c Run id (PART 2: locate the PART 1 run we are evaluating) ----------------
#
# PART 2 *consumes* the artifacts written by PART 1 (master table, best hyper-
# parameters, per-fold LOCO predictions). It must therefore not invent its own
# run id; instead it reads PART 1's run id from `artifacts/LATEST_RUN.txt`.
#
# To evaluate a *specific* historical PART 1 run rather than the latest one,
# set OVERRIDE_RUN_ID to that run's timestamp.
#
# Robust fallback:
# If LATEST_RUN points to an incomplete run (e.g., PART 1 was restarted by
# mistake), auto-fall back to the newest completed run that has:
#   - PART1_COMPLETE.txt
#   - at least one LOCO prediction parquet in predictions/

OVERRIDE_RUN_ID: str | None = None    # e.g. "20260510T120000Z"

def _looks_like_run_id(name: str) -> bool:
    return (
        len(name) == 16
        and name[8] == "T"
        and name.endswith("Z")
        and name.replace("T", "").replace("Z", "").isdigit()
    )


def _is_completed_part1_run(run_dir: Path) -> tuple[bool, str]:
    complete_file = run_dir / "PART1_COMPLETE.txt"
    pred_dir = run_dir / "predictions"
    pred_count = len(list(pred_dir.glob("*.parquet"))) if pred_dir.exists() else 0
    if not complete_file.exists():
        return False, "missing PART1_COMPLETE.txt"
    if pred_count == 0:
        return False, "no prediction parquet files"
    return True, f"ok (pred_files={pred_count})"


candidate_run_ids: list[str] = []
if OVERRIDE_RUN_ID:
    candidate_run_ids.append(OVERRIDE_RUN_ID)
if LATEST_RUN_FILE.exists():
    latest_id = LATEST_RUN_FILE.read_text().strip()
    if latest_id:
        candidate_run_ids.append(latest_id)
candidate_run_ids.extend(
    sorted(
        [p.name for p in ARTIFACTS_ROOT.iterdir() if p.is_dir() and _looks_like_run_id(p.name)],
        reverse=True,
    )
)

# de-duplicate preserving order
seen = set()
candidate_run_ids = [rid for rid in candidate_run_ids if not (rid in seen or seen.add(rid))]

RUN_TS = None
checked: list[tuple[str, str]] = []
for rid in candidate_run_ids:
    run_dir = ARTIFACTS_ROOT / rid
    ok, reason = _is_completed_part1_run(run_dir)
    checked.append((rid, reason))
    if ok:
        RUN_TS = rid
        if OVERRIDE_RUN_ID and rid == OVERRIDE_RUN_ID:
            log.info("Using user-specified OVERRIDE_RUN_ID=%s", RUN_TS)
        elif LATEST_RUN_FILE.exists() and rid == LATEST_RUN_FILE.read_text().strip():
            log.info("Using completed RUN_ID from %s -> %s", LATEST_RUN_FILE, RUN_TS)
        else:
            log.warning(
                "LATEST_RUN pointed to an incomplete run; falling back to completed run %s",
                RUN_TS,
            )
        break

if RUN_TS is None:
    details = "\n".join([f"  - {rid}: {why}" for rid, why in checked[:12]])
    raise FileNotFoundError(
        "No completed PART 1 run found for PART 2.\n"
        f"Searched under: {ARTIFACTS_ROOT}\n"
        f"Candidates checked:\n{details}\n\n"
        "Run PART 1 to completion or set OVERRIDE_RUN_ID to a completed run."
    )

RUN_DIR = ARTIFACTS_ROOT / RUN_TS
FIG_DIR = RUN_DIR / "figures"
if not RUN_DIR.exists():
    raise FileNotFoundError(
        f"Run directory does not exist: {RUN_DIR}\n"
        f"Expected this to be a PART 1 output. Did the run timestamp get re-typed?"
    )
FIG_DIR.mkdir(parents=True, exist_ok=True)
log.info("PART 2 will read PART 1 artifacts from: %s", RUN_DIR)
log.info("PART 2 will write figures/tables into : %s", FIG_DIR)

In [7]:
# --- 0.5 Master configuration -----------------------------------------------------
@dataclass(frozen=True)
class PipelineConfig:
    # ---- Targets and feature roles --------------------------------------------------
    targets: tuple[str, ...] = (
        "d18O_measurement",
        "d13C_measurement",
        "Mg_Ca_measurement",
        "Sr_Ca_measurement",
    )
    # Numeric predictors fed to the ColumnTransformer (imputed median + RobustScaler).
    # `Sr_isotopes_measurement` is intentionally OMITTED: only 164 of ~448k SISALv3
    # samples carry it, and after the row-level all-targets filter it is present
    # in <0.5 % of cohort rows. Imputing the remaining 99.5 % with the median
    # would smear a pseudo-constant across the whole feature and bias every
    # tree split / linear coefficient towards it.
    numeric_features: tuple[str, ...] = (
        "latitude",
        "longitude",
        "elevation",
        "depth_sample",
        "interp_age",
        "P_Ca_measurement",
        "U_Ca_measurement",
        "Ba_Ca_measurement",
        "age_sin",   # cyclical encoding (precessional period, see cyclical_period_kyr)
        "age_cos",
        "log_depth_sample",
        # --- Enriched per-cave covariates (built by enrich_covariates.py) ---
        "precip_d18o_bw02",   # Bowen-Wilkinson 2002 precipitation d18O isoscape -> targets d18O
        "abs_latitude",       # climate-zone proxy
        "wc_bio1", "wc_bio2", "wc_bio3", "wc_bio4", "wc_bio5", "wc_bio6", "wc_bio7",
        "wc_bio8", "wc_bio9", "wc_bio10", "wc_bio11", "wc_bio12", "wc_bio13", "wc_bio14",
        "wc_bio15", "wc_bio16", "wc_bio17", "wc_bio18", "wc_bio19",   # WorldClim bioclim
    )
    # Nominal categoricals — OneHotEncoder, *not* LabelEncoder.
    nominal_features: tuple[str, ...] = ("mineralogy", "monitoring", "hemisphere", "lat_band")
    # Identifiers that travel with X but are NEVER seen by the estimator.
    id_columns: tuple[str, ...] = ("site_id", "entity_id", "sample_id")

    # ---- Cohort filter (matches the manuscript's stated criteria) -------------------
    max_target_missing_frac: float = 0.30        # "all records missing >30% are excluded"
    require_all_four_targets_present: bool = True
    min_samples_per_site: int = 20
    mad_outlier_threshold: float = 3.0           # ">3 median absolute deviations"

    # ---- Feature engineering --------------------------------------------------------
    cyclical_period_kyr: float = 21.0            # ≈ axial precession
    high_corr_drop_threshold: float = 0.95       # applied train-fold-only inside the pipeline

    # ---- Splitting ------------------------------------------------------------------
    inner_cv_groupkfold_splits: int = 5          # legacy: used only by tune_tabular (§3.5, unused); §3.7 uses GroupKFold(3)
    block_gap_fraction: float = 0.10             # within-cave block gap = 10 % of entity length
    block_gap_min_samples: int = 30
    min_entity_length_for_blockgap: int = 80

    # ---- Hyperparameter search ------------------------------------------------------
    rf_param_distributions: dict = field(
        default_factory=lambda: {
            "estimator__n_estimators": [200, 400, 600, 800],
            "estimator__max_depth": [None, 8, 12, 16, 24],
            "estimator__min_samples_split": [2, 5, 10],
            "estimator__min_samples_leaf": [1, 2, 4],
            "estimator__max_features": ["sqrt", 0.5, 1.0],
        }
    )
    xgb_param_distributions: dict = field(
        default_factory=lambda: {
            "estimator__n_estimators": [300, 500, 800, 1200],
            "estimator__max_depth": [4, 6, 8, 10, 12],
            "estimator__learning_rate": [0.02, 0.05, 0.08, 0.12],
            "estimator__subsample": [0.6, 0.8, 1.0],
            "estimator__colsample_bytree": [0.6, 0.8, 1.0],
            "estimator__reg_alpha": [0.0, 0.01, 0.1],
            "estimator__reg_lambda": [0.5, 1.0, 2.0],
            "estimator__min_child_weight": [1, 3, 5],
        }
    )
    # 20 iters keeps each (model, target) tabular search ~3-5 min on Colab Pro;
    # bump back to 30+ for a paper-camera-ready run if reviewers ask.
    n_random_search_iter: int = 20

    # ---- Sequence DL ----------------------------------------------------------------
    sequence_length: int = 10
    sequence_stride: int = 1
    bilstm_units_grid: tuple[int, ...] = (32, 64, 128)
    bilstm_dropout_grid: tuple[float, ...] = (0.1, 0.2, 0.3)
    bilstm_layers_grid: tuple[int, ...] = (1, 2)
    transformer_dmodel_grid: tuple[int, ...] = (64, 128)
    transformer_heads_grid: tuple[int, ...] = (2, 4)
    transformer_ff_grid: tuple[int, ...] = (128, 256)
    transformer_layers_grid: tuple[int, ...] = (1, 2)
    dl_batch_size: int = 64
    dl_max_epochs: int = 60
    dl_early_stop_patience: int = 8
    # 6 candidates per (DL_model, target) keeps each DL search ~10-20 min on a
    # Colab Pro T4/V100; raise for a paper-camera-ready run.
    dl_n_random_search_iter: int = 6

    # ---- Evaluation -----------------------------------------------------------------
    # 500 bootstrap resamples gives ~1% CI-width resolution at n=41 folds — more
    # than enough for the manuscript's reported precision. Raise to 1000 for the
    # final camera-ready CIs if a reviewer specifically asks.
    bootstrap_resamples: int = 500
    # 50 iters gives p-value resolution of 0.02 — sufficient for the standard
    # "p<0.05" reporting threshold and keeps each permutation test under ~2 h
    # on Colab Pro. Bump back to 100+ for a final camera-ready run.
    permutation_n_iter: int = 50
    # Restrict the (very expensive) permutation test to the single best-RMSE
    # model per target by default. Set to False to run it for every tabular
    # model (warning: ~5x more compute, easily blows past Colab Pro's session cap).
    permutation_only_best_per_target: bool = True

    # ---- Determinism ----------------------------------------------------------------
    random_seed: int = 42

    # ---- Diagnostic toggles ---------------------------------------------------------
    fast_mode: bool = False                      # if True, shrink iterations for smoke-tests


CONFIG = PipelineConfig()

if globals().get("LOCAL_PROFILE", False):
    CONFIG = replace(
        CONFIG,
        n_random_search_iter=10,
        dl_n_random_search_iter=2,
        dl_max_epochs=25,
        bootstrap_resamples=200,
        permutation_n_iter=20,
        fast_mode=True,
    )
    log.warning(
        "LOCAL_PROFILE is ON: lighter defaults applied "
        "(search iters/epochs/bootstrap/permutation reduced)."
    )

if CONFIG.fast_mode:
    log.warning("CONFIG.fast_mode is ON — iteration counts will be reduced for debugging.")

# Persist config alongside artifacts so the run is reproducible.
_cfg_dump = {
    k: (list(v) if isinstance(v, tuple) else v)
    for k, v in CONFIG.__dict__.items()
}
(RUN_DIR / "config.json").write_text(json.dumps(_cfg_dump, indent=2, default=str))
log.info("Configuration persisted to %s", RUN_DIR / "config.json")

In [8]:
# --- 0.6 Determinism --------------------------------------------------------------
def set_global_determinism(seed: int) -> None:
    """Seed every stochastic library used downstream.

    Note: full bit-for-bit reproducibility on GPU also requires
    ``TF_DETERMINISTIC_OPS=1`` *before* TensorFlow is imported. We set it here
    pre-emptively so the first ``import tensorflow`` later picks it up.
    """
    os.environ["PYTHONHASHSEED"] = str(seed)
    os.environ.setdefault("TF_DETERMINISTIC_OPS", "1")
    os.environ.setdefault("TF_CUDNN_DETERMINISTIC", "1")
    random.seed(seed)
    np.random.seed(seed)


set_global_determinism(CONFIG.random_seed)
log.info("Global RNGs seeded with %d", CONFIG.random_seed)

---

## 1. Reload PART 1 artifacts

> **PART 2 of 2** — *evaluation & visualisation notebook.* This notebook reloads the master table, best hyper-parameters and per-fold LOCO predictions written by PART 1 (`001_SISALv3_Pipeline_PART1_Train_ENRICHED.ipynb`), then runs the within-cave gap-fill experiment (Section 5), the optional permutation tests (Section 4.5), the pairwise paired-Wilcoxon tests (Section 4.6), and the entire publication plotting suite (Section 6). Typical wall-time on Colab Pro is **~3–6 h** with default settings.

The cell below re-loads the cohort produced by PART 1's master-table-assembly step (including enriched per-cave covariates). We then re-derive the canonical helper arrays (`X_features`, `groups_site`, `groups_entity`, `sample_ids`, `site_names`) so every downstream cell sees exactly the same row ordering and feature columns PART 1 used.

In [9]:
# --- 1. Load master_table.parquet and rebuild canonical helper arrays -------------
MASTER_PARQUET = DATA_OUT_DIR / "master_table.parquet"
COHORT_CSV = RUN_DIR / "cohort.csv"

if not MASTER_PARQUET.exists():
    raise FileNotFoundError(
        f"master_table.parquet not found at {MASTER_PARQUET}.\n"
        f"Run PART 1 of the pipeline first; it writes this file."
    )
master_table = pd.read_parquet(MASTER_PARQUET)
log.info("Loaded master_table : rows=%d cols=%d sites=%d entities=%d",
         len(master_table), master_table.shape[1],
         master_table["site_id"].nunique(), master_table["entity_id"].nunique())

if COHORT_CSV.exists():
    cohort_audit = pd.read_csv(COHORT_CSV)
    log.info("Loaded cohort_audit : %d sites", len(cohort_audit))
else:
    log.warning("cohort.csv missing; rebuilding a minimal audit table from master_table")
    cohort_audit = (
        master_table.groupby(["site_id", "site_name"], dropna=False)
        .agg(
            n_entities=("entity_id", "nunique"),
            n_samples=("sample_id", "size"),
            latitude=("latitude", "first"),
            longitude=("longitude", "first"),
            elevation=("elevation", "first"),
            age_min_yr_BP=("interp_age", "min"),
            age_max_yr_BP=("interp_age", "max"),
        )
        .reset_index()
        .sort_values("n_samples", ascending=False)
    )

# --- Ensure enriched per-cave covariates match PART 1 ENRICHED --------------------
# PART 1 merges these into master_table before writing master_table.parquet; if an
# older PART 1 run omitted them, fall back to master_table_enriched.parquet.
_COVARIATES = [
    "precip_d18o_bw02", "abs_latitude", "hemisphere", "lat_band",
    *[f"wc_bio{b}" for b in range(1, 20)],
]
_ENR_PATH = DATA_OUT_DIR / "master_table_enriched.parquet"
_missing_cov = [c for c in _COVARIATES if c not in master_table.columns]
if _missing_cov:
    if _ENR_PATH.exists():
        _enr = pd.read_parquet(_ENR_PATH)
        _have = [c for c in _missing_cov if c in _enr.columns]
        if _have:
            _per_cave = _enr.groupby("site_id")[_have].first().reset_index()
            master_table = master_table.merge(_per_cave, on="site_id", how="left")
            log.info("Merged %d enriched covariates from %s", len(_have), _ENR_PATH.name)
            print(f"Merged {len(_have)} enriched covariates from {_ENR_PATH.name}")
        _still_missing = [c for c in _COVARIATES if c not in master_table.columns]
        if _still_missing:
            log.warning("Enriched columns still missing after merge: %s", _still_missing)
            print(f"WARNING: still missing enriched columns: {_still_missing}")
    else:
        log.warning(
            "Enriched columns missing from master_table and %s not found: %s",
            _ENR_PATH, _missing_cov,
        )
        print(f"WARNING: enriched covariates missing ({_missing_cov[:5]}...). "
              "Re-run PART 1 ENRICHED or enrich_covariates.py.")
else:
    _cov = master_table[_COVARIATES].notna().all(axis=1).mean()
    log.info("Enriched covariates present in master_table; full-coverage rows: %.1f%%", 100 * _cov)
    print(f"Enriched covariates OK. Coverage: {_cov:.1%} of rows have all of them.")

_model_feats = list(CONFIG.numeric_features) + list(CONFIG.nominal_features)
_absent = [c for c in _model_feats if c not in master_table.columns]
if _absent:
    raise ValueError(
        f"CONFIG features missing from master_table: {_absent}. "
        "PART 2 CONFIG must match PART 1 ENRICHED."
    )

X_features    = master_table.drop(columns=list(CONFIG.targets))
groups_site   = master_table["site_id"].values
groups_entity = master_table["entity_id"].values
sample_ids    = master_table["sample_id"].values
site_names    = master_table["site_name"].values

Merged 23 enriched covariates from master_table_enriched.parquet


---

## 2. Cave-aware splitters and leakage-safe preprocessing

This section builds the four reusable primitives that every model in the pipeline depends on:

| Primitive | What it does | Why it matters |
|-----------|--------------|----------------|
| `LeaveOneCaveOut` | Standard `sklearn.model_selection.LeaveOneGroupOut` keyed on `site_id`. | The headline experiment of the paper (Tables 1–3). |
| `BlockedWithinCaveSplitter` | For each `entity_id`, mask a contiguous, age-sorted block of size ≈ `block_gap_fraction × len(entity)` as the test set; train on the rest of *that* entity *plus* every other entity. | The secondary "gap-fill" experiment (Table 4) that justifies cubic spline as a baseline. |
| `make_column_transformer_factory` | Returns a fresh `ColumnTransformer` per fold: median impute + `RobustScaler` for numerics, most-frequent impute + `OneHotEncoder(handle_unknown="infrequent_if_exist")` for nominals. | All scaling/imputation statistics are fit *only on the training fold*. |
| `CorrelationDropper` | sklearn-compatible transformer that, in `fit`, looks at the training-fold-only correlation matrix and removes one feature from each pair with `\|ρ\| ≥ cfg.high_corr_drop_threshold`. | Matches the paper's claim of redundant-feature removal *without* the leakage of computing correlations across the entire dataset. |

A small `summarize_splits` helper prints the number of folds, hold-out cave name, and train/test row counts so the splits are auditable in the notebook output.

In [10]:
from sklearn.model_selection import LeaveOneGroupOut
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import RobustScaler, OneHotEncoder

# --- 2.1 LeaveOneCaveOut ----------------------------------------------------------
# We thinly wrap LeaveOneGroupOut so it has a name that matches the manuscript
# vocabulary ("Leave-One-Cave-Out") and so we can attach a `summarize_splits`
# convenience method.
class LeaveOneCaveOut(LeaveOneGroupOut):
    """LOGO keyed on `site_id`. Sentinel name to match the manuscript wording."""

    def get_n_splits(self, X=None, y=None, groups=None):
        return super().get_n_splits(X, y, groups)


def summarize_splits(splitter, X, groups, name: str) -> pd.DataFrame:
    """Tabulate every fold's train/test row count and the held-out group label.

    Returns the audit DataFrame and also writes it to RUN_DIR/{name}_splits.csv
    so it lands in the per-run artifact bundle.
    """
    rows = []
    for k, (tr, te) in enumerate(splitter.split(X, groups=groups)):
        held_out = sorted(set(groups[te]))
        rows.append({
            "fold": k,
            "held_out_group": held_out[0] if len(held_out) == 1 else f"{len(held_out)} groups",
            "n_train": len(tr),
            "n_test": len(te),
            "frac_test": len(te) / (len(tr) + len(te)),
        })
    audit = pd.DataFrame(rows)
    out_path = RUN_DIR / f"{name}_splits.csv"
    audit.to_csv(out_path, index=False)
    log.info("Splitter [%s] : %d folds (audit -> %s)", name, len(audit), out_path)
    return audit

In [11]:
# --- 2.2 BlockedWithinCaveSplitter ------------------------------------------------
import zlib


class BlockedWithinCaveSplitter:
    """Within-entity block-gap splitter for the secondary gap-fill experiment.

    For each entity that meets `min_entity_length`, we mark a single contiguous
    test block of length `max(min_block_samples, ceil(block_fraction * n))`
    centred at a deterministic offset (different per entity, derived from
    `random_state` and `entity_id` for reproducibility). The training set for
    every fold consists of:

      - every row of every *other* entity, AND
      - the rows of *this* entity that fall outside the test block.

    This makes cubic-spline / linear interpolation a meaningful baseline (the
    interpolator can use the temporally adjacent training rows of the same
    entity to bridge the gap), while still being a real prediction problem
    (the test rows themselves are unseen during fitting).

    Reproducibility (revision, Nov 2026)
    ------------------------------------
    The original version seeded each block with Python's `hash()`, which is
    randomised per interpreter session, so reruns placed the blocks differently
    from the run behind the published tables. Two fixes:

    - `fixed_blocks`: path to `wcbg_heldout_blocks.csv` (columns entity_id,
      sample_id), exported from the published run. With it (and `sample_ids`
      aligned with the rows of X), the splitter returns exactly the published
      46 blocks, in the same fold order. This is the default below whenever the
      CSV is found.
    - Without `fixed_blocks`, block positions are seeded with `zlib.crc32`,
      which is stable across sessions and machines (but gives different blocks
      from the published run).

    Notes
    -----
    - Sequences for the BiLSTM/Transformer should be sliced *per entity*; the
      sequence-builder elsewhere in the notebook will skip windows that overlap
      the test block.
    - Returned indices index into the *position-aligned* arrays passed to
      `split(X, y, groups)`. We expect `groups` to be the entity_id column.
    """

    def __init__(
        self,
        block_fraction: float,
        min_block_samples: int,
        min_entity_length: int,
        random_state: int = 42,
        fixed_blocks=None,
        sample_ids=None,
    ):
        self.block_fraction = float(block_fraction)
        self.min_block_samples = int(min_block_samples)
        self.min_entity_length = int(min_entity_length)
        self.random_state = int(random_state)
        self.fixed_blocks = fixed_blocks
        self.sample_ids = sample_ids

    def _load_fixed(self):
        if self.fixed_blocks is None:
            return None
        fb = self.fixed_blocks if isinstance(self.fixed_blocks, pd.DataFrame) else pd.read_csv(self.fixed_blocks)
        return fb[["entity_id", "sample_id"]]

    def _seed(self, ent) -> int:
        return zlib.crc32(f"{self.random_state}|{ent}".encode()) & 0xFFFFFFFF

    def get_n_splits(self, X=None, y=None, groups=None) -> int:
        fb = self._load_fixed()
        if fb is not None:
            return int(fb["entity_id"].nunique())
        if groups is None:
            return 0
        groups = pd.Series(groups).reset_index(drop=True)
        ent_sizes = groups.value_counts()
        return int((ent_sizes >= self.min_entity_length).sum())

    def split(self, X, y=None, groups=None):
        if groups is None:
            raise ValueError("BlockedWithinCaveSplitter requires `groups` (entity_id).")
        n = len(groups)
        groups = pd.Series(groups).reset_index(drop=True)

        fb = self._load_fixed()
        if fb is not None:
            if self.sample_ids is None or len(self.sample_ids) != n:
                raise ValueError("fixed_blocks requires `sample_ids` aligned with the rows of X.")
            sid = pd.Series(np.asarray(self.sample_ids))
            for ent, g in fb.groupby("entity_id", sort=True):
                test_idx = np.where(sid.isin(set(g["sample_id"])).to_numpy())[0]
                if len(test_idx) != len(g):
                    raise ValueError(f"entity {ent}: {len(g)} block samples in fixed_blocks, {len(test_idx)} found in X")
                if not (groups.values[test_idx] == ent).all():
                    raise ValueError(f"entity {ent}: fixed-block samples belong to another entity in `groups`")
                mask = np.ones(n, dtype=bool)
                mask[test_idx] = False
                yield np.where(mask)[0], test_idx
            return

        # Eligible entities = those with enough samples to host a meaningful gap.
        eligible = (
            groups.value_counts()
                  .loc[lambda s: s >= self.min_entity_length]
                  .index
                  .tolist()
        )
        for ent in sorted(eligible):
            idx = np.where(groups.values == ent)[0]
            n_ent = len(idx)
            block_size = max(self.min_block_samples, int(np.ceil(self.block_fraction * n_ent)))
            block_size = min(block_size, n_ent - 4)  # leave at least a few train rows in this entity
            # Deterministic centre per entity, stable across sessions (crc32, not hash()).
            rs = np.random.default_rng(self._seed(ent))
            margin = max(2, n_ent // 6)
            start_lo = margin
            start_hi = max(start_lo + 1, n_ent - block_size - margin)
            start = int(rs.integers(start_lo, start_hi)) if start_hi > start_lo else start_lo
            test_local = np.arange(start, start + block_size)
            test_idx = idx[test_local]
            mask = np.ones(n, dtype=bool)
            mask[test_idx] = False
            train_idx = np.where(mask)[0]
            yield train_idx, test_idx


# Exact held-out blocks behind the published within-cave results (Table 2 of the paper).
_WCBG_CANDIDATES = [Path("wcbg_heldout_blocks.csv"), Path("data/wcbg_heldout_blocks.csv"),
                    Path("../data/wcbg_heldout_blocks.csv"), PROJECT_ROOT / "wcbg_heldout_blocks.csv",
                    PROJECT_ROOT / "data" / "wcbg_heldout_blocks.csv",
                    PROJECT_ROOT / "release_github" / "data" / "wcbg_heldout_blocks.csv"]
WCBG_BLOCKS_CSV = next((p for p in _WCBG_CANDIDATES if p.exists()), None)
if WCBG_BLOCKS_CSV is None:
    log.warning("wcbg_heldout_blocks.csv not found (looked in %s); within-cave blocks will be placed with the "
                "crc32 seed and will differ from the published run.", [str(p) for p in _WCBG_CANDIDATES])
else:
    log.info("Within-cave blocks: using the published blocks from %s", WCBG_BLOCKS_CSV)

In [12]:
# --- 2.3 CorrelationDropper -------------------------------------------------------
class CorrelationDropper(BaseEstimator, TransformerMixin):
    """Drop one feature from each highly-correlated pair, fit per train fold.

    Operates on the *output* of the numeric arm of the ColumnTransformer (so it
    sees scaled, imputed numeric features only). The decision to drop is made
    using the training fold's correlation matrix only — never the held-out
    fold — which keeps the transformation leakage-safe.

    The first feature in each highly-correlated pair (column-order tie-break)
    is retained; the second is dropped. If you want different priorities,
    reorder `numeric_features` in CONFIG.
    """

    def __init__(self, threshold: float = 0.95):
        self.threshold = float(threshold)

    def fit(self, X, y=None):
        Xa = np.asarray(X)
        if Xa.ndim != 2:
            raise ValueError(f"CorrelationDropper expects 2-D input, got {Xa.shape}")
        self.n_features_in_ = Xa.shape[1]
        if Xa.shape[0] < 4 or Xa.shape[1] < 2:
            self.kept_mask_ = np.ones(self.n_features_in_, dtype=bool)
            self.dropped_indices_ = np.array([], dtype=int)
            return self
        # nan-safe Pearson correlation
        with np.errstate(invalid="ignore", divide="ignore"):
            corr = np.corrcoef(Xa, rowvar=False)
        corr = np.where(np.isfinite(corr), corr, 0.0)
        upper = np.triu(np.abs(corr), k=1)
        to_drop: set[int] = set()
        for i in range(corr.shape[0]):
            for j in range(i + 1, corr.shape[1]):
                if upper[i, j] >= self.threshold and j not in to_drop:
                    to_drop.add(j)
        self.kept_mask_ = np.array(
            [k not in to_drop for k in range(self.n_features_in_)], dtype=bool
        )
        self.dropped_indices_ = np.array(sorted(to_drop), dtype=int)
        return self

    def transform(self, X):
        Xa = np.asarray(X)
        return Xa[:, self.kept_mask_]

    def get_feature_names_out(self, input_features=None):
        if input_features is None:
            input_features = np.array([f"x{i}" for i in range(self.n_features_in_)])
        input_features = np.asarray(input_features)
        return input_features[self.kept_mask_]

In [13]:
# --- 2.4 ColumnTransformer factory ------------------------------------------------
def make_column_transformer_factory(
    numeric_features: list[str],
    nominal_features: list[str],
    drop_corr_threshold: float | None = 0.95,
) -> Callable[[], Pipeline]:
    """Return a *factory* that builds a fresh, leakage-safe preprocessor each call.

    Why a factory? RandomizedSearchCV refits the pipeline for every candidate /
    fold; we need a brand-new ColumnTransformer per call so no cross-fold state
    can ever be re-used.

    Numeric arm   : SimpleImputer(median) → RobustScaler() → optional CorrelationDropper.
    Nominal arm   : SimpleImputer(most_frequent) → OneHotEncoder(handle_unknown=
                    "infrequent_if_exist", min_frequency=2, sparse_output=False).
    """

    def _build() -> Pipeline:
        numeric_pipe_steps = [
            ("impute", SimpleImputer(strategy="median")),
            ("scale", RobustScaler()),
        ]
        if drop_corr_threshold is not None:
            numeric_pipe_steps.append(("drop_corr", CorrelationDropper(drop_corr_threshold)))
        numeric_pipe = Pipeline(numeric_pipe_steps)

        nominal_pipe = Pipeline(
            [
                ("impute", SimpleImputer(strategy="most_frequent")),
                (
                    "ohe",
                    OneHotEncoder(
                        handle_unknown="infrequent_if_exist",
                        min_frequency=2,
                        sparse_output=False,
                    ),
                ),
            ]
        )

        ct = ColumnTransformer(
            transformers=[
                ("num", numeric_pipe, numeric_features),
                ("cat", nominal_pipe, nominal_features),
            ],
            remainder="drop",
            verbose_feature_names_out=False,
        )
        return Pipeline([("preprocess", ct)])

    return _build

In [14]:
# --- 2.5 Quick sanity-check: instantiate the splitters and the preprocessor ------
loco = LeaveOneCaveOut()
within = BlockedWithinCaveSplitter(
    block_fraction=CONFIG.block_gap_fraction,
    min_block_samples=CONFIG.block_gap_min_samples,
    min_entity_length=CONFIG.min_entity_length_for_blockgap,
    random_state=CONFIG.random_seed,
    fixed_blocks=WCBG_BLOCKS_CSV,
    sample_ids=master_table["sample_id"].values,
)

X_features = master_table.drop(columns=list(CONFIG.targets))
groups_site = master_table["site_id"].values
groups_entity = master_table["entity_id"].values

loco_audit = summarize_splits(loco, X_features, groups_site, name="loco")
within_audit = summarize_splits(within, X_features, groups_entity, name="within_cave_block")

print("\nLOCO splits (head):")
print(loco_audit.head(8).to_string(index=False))
print(f"\n... total {len(loco_audit)} folds; mean test fraction = "
      f"{loco_audit['frac_test'].mean():.3f}, "
      f"min/max test rows = {loco_audit['n_test'].min()}/{loco_audit['n_test'].max()}")

print("\nWithin-cave block-gap splits (head):")
print(within_audit.head(8).to_string(index=False))
print(f"\n... total {len(within_audit)} folds (one per eligible entity)")

# Build the preprocessor factory once; every model below will instantiate
# a fresh copy via this factory inside its sklearn pipeline.
make_preprocessor = make_column_transformer_factory(
    numeric_features=list(CONFIG.numeric_features),
    nominal_features=list(CONFIG.nominal_features),
    drop_corr_threshold=CONFIG.high_corr_drop_threshold,
)

# Fit it on the FULL cohort once just to verify it works end-to-end (this is
# *not* used downstream — every CV fold builds and fits its own copy).
_demo_pre = make_preprocessor()
_demo_X = _demo_pre.fit_transform(X_features.head(2000))
log.info(
    "Preprocessor demo fit OK: input cols=%d, output cols=%d (after impute+scale+OHE+corr-drop)",
    X_features.shape[1], _demo_X.shape[1],
)


LOCO splits (head):
 fold  held_out_group  n_train  n_test  frac_test
    0               2    17460     544   0.030216
    1               4    17888     116   0.006443
    2              14    17932      72   0.003999
    3              29    17192     812   0.045101
    4              70    16836    1168   0.064874
    5              79    17486     518   0.028771
    6             104    17770     234   0.012997
    7             117    17000    1004   0.055765

... total 41 folds; mean test fraction = 0.024, min/max test rows = 27/1880

Within-cave block-gap splits (head):
 fold  held_out_group  n_train  n_test  frac_test
    0              17    17949      55   0.003055
    1              32    17974      30   0.001666
    2             101    17974      30   0.001666
    3             102    17943      61   0.003388
    4             156    17974      30   0.001666
    5             158    17974      30   0.001666
    6             169    17963      41   0.002277
    7         

---

## 3. Models and per-target hyper-parameter search

We train **five** model families, each fitted **per target** (one estimator per `(model, target)` pair) so that hyper-parameters can be optimized for the very different distributions of `d18O`, `d13C`, `Mg/Ca`, and `Sr/Ca`:

| Model | Inductive bias | Why it's in the panel |
|-------|----------------|-----------------------|
| **Random Forest** | Bagging of unconstrained trees, axis-aligned splits. | Strong, low-variance baseline; matches the legacy notebook's headline. |
| **XGBoost** | Gradient boosting, asymmetric trees, L1+L2 leaf regularization. | Classical SOTA tabular boosting baseline. |
| **CatBoost** | Ordered boosting + symmetric (oblivious) trees + native categorical splits. | Reduces target leakage during training, designed to generalize to unseen distributions — the exact failure mode LOCO measures. |
| **BiLSTM** (sequence) | Bidirectional recurrent network over chronological windows of 10 consecutive samples per entity. | Tests whether *temporal context within a stalagmite* improves cross-cave prediction. |
| **Transformer** (sequence) | Self-attention over the same per-entity windows, with a learned positional embedding for `interp_age`. | Same hypothesis as BiLSTM but with a global-attention inductive bias instead of recurrence. |

### Hyper-parameter search strategy

For tractability, we run **one** `HalvingRandomSearchCV` per `(model, target)` pair for the tree ensembles (RF, XGBoost, CatBoost) on the *full* cohort (§3.7): 20 initial candidates, `factor=3`, `min_resources` = 40% of samples, `aggressive_elimination=True` (three halving rounds), scored by RMSE with `GroupKFold(3)` keyed on `site_id` (so the inner CV is itself cave-aware — selected hyper-parameters generalize across caves rather than over-fitting any single one). The best parameters are then **frozen** and applied to every fold of the outer LOCO loop. This is a common simplification (SearchCV scoring sees the full cohort, but no model *weights* ever cross fold boundaries) — strictly cleaner-than-nested LOCO would multiply compute by ~7× without changing the qualitative story.

For the deep-learning models we run a smaller, hand-rolled random search over six candidate architectures (`dl_n_random_search_iter=6`) with the same cave-aware `GroupKFold(3)` inner CV, because each candidate involves a full TensorFlow training loop. The search is run for the first target (δ¹⁸O), and the selected architecture is reused for the other three targets (`arch_reused_from` in the saved JSON).

The `tune_tabular` helper in §3.5 (`RandomizedSearchCV`) is kept for reference only; the published hyper-parameters were produced by §3.7.

All best-hyper-parameter dictionaries are persisted to `artifacts/<run-id>/best_params/<model>_<target>.json`.

In [15]:
# --- 3.1 Tabular model factories --------------------------------------------------
import xgboost as xgb
from sklearn.ensemble import RandomForestRegressor

try:
    from catboost import CatBoostRegressor
    HAS_CATBOOST = True
except ImportError:
    log.warning("catboost is not installed; skipping the CatBoost panel entry.")
    HAS_CATBOOST = False


def make_rf_pipeline() -> Pipeline:
    """Random Forest on the leakage-safe preprocessor."""
    pre = make_preprocessor()
    rf = RandomForestRegressor(
        n_estimators=400,
        random_state=CONFIG.random_seed,
        n_jobs=-1,
    )
    return Pipeline([("preprocess", pre.named_steps["preprocess"]),
                     ("estimator", rf)])


def make_xgb_pipeline() -> Pipeline:
    """XGBoost on the leakage-safe preprocessor."""
    pre = make_preprocessor()
    estimator = xgb.XGBRegressor(
        n_estimators=500,
        learning_rate=0.05,
        max_depth=6,
        random_state=CONFIG.random_seed,
        tree_method="hist",
        n_jobs=-1,
        objective="reg:squarederror",
    )
    return Pipeline([("preprocess", pre.named_steps["preprocess"]),
                     ("estimator", estimator)])


def make_catboost_pipeline() -> Pipeline:
    """CatBoost on the leakage-safe preprocessor.

    NOTE: We feed CatBoost the same OHE-expanded matrix as the other models
    rather than its native categorical columns, so the *only* differences
    between the panel members are the learning algorithm itself, not the
    feature representation. This makes the per-model comparison cleaner.
    """
    if not HAS_CATBOOST:
        raise RuntimeError("catboost not installed")
    pre = make_preprocessor()
    estimator = CatBoostRegressor(
        iterations=600,
        depth=6,
        learning_rate=0.05,
        l2_leaf_reg=3.0,
        random_seed=CONFIG.random_seed,
        verbose=False,
        allow_writing_files=False,
        thread_count=-1,
    )
    return Pipeline([("preprocess", pre.named_steps["preprocess"]),
                     ("estimator", estimator)])

In [16]:
# --- 3.2 Sequence builder (per entity, age-sorted, no boundary crossing) ----------
def build_sequences_per_entity(
    X_proc: np.ndarray,
    y: np.ndarray | None,
    entity_ids: np.ndarray,
    sequence_length: int,
    stride: int = 1,
) -> tuple[np.ndarray, np.ndarray | None, np.ndarray]:
    """Slice (X_proc, y) into fixed-length windows that never cross entity boundaries.

    The input rows must already be sorted by ``(entity_id, interp_age)`` —
    which is exactly the canonical sort applied in `apply_cohort_filter`.

    For an entity of length n_e:
      - if n_e >= sequence_length: produce ``floor((n_e - L) / stride) + 1`` windows,
        each of shape (L, n_features). The label associated with each window is
        the y-value at its LAST row (the standard "predict-current-given-past+
        present-context" framing).
      - if n_e < sequence_length: produce one *left-padded* window where the
        first ``L - n_e`` rows are zero-vectors. This way every entity
        contributes at least one prediction at inference time.

    Returns
    -------
    X_seq : (n_windows, L, n_features) float32
    y_seq : (n_windows,) float32, or None if y is None
    last_row_idx : (n_windows,) int — the index into the *original* X_proc
                   array of each window's labelled (last) row, used at
                   inference time to project predictions back to row positions.
    """
    X_proc = np.asarray(X_proc, dtype=np.float32)
    if y is not None:
        y = np.asarray(y, dtype=np.float32)
    entity_ids = np.asarray(entity_ids)

    L = int(sequence_length)
    s = int(stride)
    n_features = X_proc.shape[1]

    seq_X: list[np.ndarray] = []
    seq_y: list[float] = []
    seq_idx: list[int] = []

    # Group by entity preserving the order rows arrive in (already age-sorted).
    pos = 0
    for ent in pd.unique(entity_ids):
        mask = entity_ids == ent
        idx = np.where(mask)[0]
        n_e = len(idx)
        if n_e == 0:
            continue
        Xe = X_proc[idx]

        if n_e < L:
            # Left-pad with zeros so the entity still produces one prediction
            # (only its terminal row).
            pad = np.zeros((L - n_e, n_features), dtype=np.float32)
            window = np.vstack([pad, Xe])
            seq_X.append(window)
            seq_idx.append(int(idx[-1]))
            if y is not None:
                seq_y.append(float(y[idx[-1]]))
        else:
            for start in range(0, n_e - L + 1, s):
                end = start + L
                seq_X.append(Xe[start:end])
                seq_idx.append(int(idx[end - 1]))
                if y is not None:
                    seq_y.append(float(y[idx[end - 1]]))

    X_arr = np.stack(seq_X, axis=0).astype(np.float32) if seq_X else np.zeros((0, L, n_features), dtype=np.float32)
    y_arr = np.asarray(seq_y, dtype=np.float32) if y is not None else None
    idx_arr = np.asarray(seq_idx, dtype=np.int64)
    return X_arr, y_arr, idx_arr

In [17]:
# --- 3.3 SequenceWrapper: sklearn-compatible wrapper around the per-entity DL flow
class SequenceWrapper(BaseEstimator):
    """Wrap a Keras model factory in an sklearn-style fit/predict surface.

    Key design points (no sliding-window-across-cave-boundaries hacks):

      - `fit(X, y, entity_ids)` and `predict(X, entity_ids)` *require* an
        `entity_ids` array; this is the only way to honor per-entity
        chronological ordering. (We can't rely on sklearn's grouping protocol
        because it isn't threaded through pipelines for inference.)
      - The wrapped model emits one scalar per window; we lift that back to
        one prediction per *row* via `last_row_idx` from `build_sequences_*`.
      - Rows that aren't the terminal row of any window receive the
        prediction of the most-recent window terminating at-or-after them
        (forward-fill within their entity). This guarantees every row has a
        prediction even at the very start of an entity (where left-padding
        kicks in).
    """

    def __init__(
        self,
        model_factory: Callable[[int, int], Any],
        sequence_length: int,
        n_features: int,
        batch_size: int = 64,
        max_epochs: int = 60,
        early_stop_patience: int = 8,
        verbose: int = 0,
    ):
        self.model_factory = model_factory
        self.sequence_length = int(sequence_length)
        self.n_features = int(n_features)
        self.batch_size = int(batch_size)
        self.max_epochs = int(max_epochs)
        self.early_stop_patience = int(early_stop_patience)
        self.verbose = int(verbose)

    def fit(self, X, y, entity_ids):
        # Lazy TF imports so the notebook can be inspected without TF available.
        import tensorflow as tf
        from tensorflow.keras import callbacks

        X_seq, y_seq, _ = build_sequences_per_entity(
            X, y, entity_ids,
            sequence_length=self.sequence_length,
            stride=1,
        )
        if X_seq.shape[0] == 0:
            raise ValueError("SequenceWrapper.fit produced 0 windows — check entity_ids alignment")

        # Re-seed TF per fit so repeated calls (e.g. inside hyper-search) are deterministic.
        tf.random.set_seed(CONFIG.random_seed)
        self.model_ = self.model_factory(self.sequence_length, self.n_features)
        cbs = [
            callbacks.EarlyStopping(
                monitor="loss",
                patience=self.early_stop_patience,
                restore_best_weights=True,
            )
        ]
        self.history_ = self.model_.fit(
            X_seq, y_seq,
            batch_size=self.batch_size,
            epochs=self.max_epochs,
            verbose=self.verbose,
            callbacks=cbs,
        ).history
        self.n_features_in_ = self.n_features
        return self

    def predict(self, X, entity_ids):
        if not hasattr(self, "model_"):
            raise RuntimeError("SequenceWrapper.predict called before fit()")
        X_seq, _, last_idx = build_sequences_per_entity(
            X, None, entity_ids,
            sequence_length=self.sequence_length,
            stride=1,
        )
        if X_seq.shape[0] == 0:
            return np.zeros(len(X), dtype=np.float32)

        y_seq_pred = self.model_.predict(X_seq, batch_size=self.batch_size, verbose=0).ravel()

        # Project window-level predictions back to row-level. For each row,
        # take the prediction of the *first* window whose terminal row is at or
        # after this row, scoped to the row's own entity.
        n = len(X)
        y_pred = np.full(n, np.nan, dtype=np.float32)
        # First, place each window's prediction at its terminal row.
        y_pred[last_idx] = y_seq_pred
        # Forward-fill within each entity for rows that weren't anyone's terminal row.
        for ent in pd.unique(entity_ids):
            mask = entity_ids == ent
            ent_idx = np.where(mask)[0]
            ent_preds = y_pred[ent_idx]
            # back-then-forward fill so leading NaNs are filled by the first
            # available terminal-row prediction in that entity.
            s = pd.Series(ent_preds)
            s = s.bfill().ffill()
            y_pred[ent_idx] = s.values
        # Anything still NaN (shouldn't happen): fall back to overall mean of
        # window predictions, which keeps the metric well-defined.
        if np.isnan(y_pred).any():
            y_pred = np.where(np.isnan(y_pred), float(np.nanmean(y_seq_pred)), y_pred)
        return y_pred

In [18]:
# --- 3.4 BiLSTM and Transformer Keras model factories -----------------------------
def build_bilstm_model(
    sequence_length: int,
    n_features: int,
    units: int = 64,
    n_layers: int = 1,
    dropout: float = 0.2,
    learning_rate: float = 1e-3,
):
    """Build a BiLSTM regressor that emits one scalar per window.

    The same architecture is used for every target — we tune (`units`,
    `n_layers`, `dropout`) per target via the random search.
    """
    import tensorflow as tf
    from tensorflow.keras import layers, models, optimizers

    inputs = layers.Input(shape=(sequence_length, n_features), name="input")
    x = inputs
    for i in range(n_layers):
        return_sequences = i < n_layers - 1  # last layer collapses to a vector
        x = layers.Bidirectional(
            layers.LSTM(units, return_sequences=return_sequences, dropout=dropout)
        )(x)
    if n_layers == 0:
        x = layers.Flatten()(x)
    x = layers.Dense(units, activation="relu")(x)
    x = layers.Dropout(dropout)(x)
    outputs = layers.Dense(1, name="y")(x)
    model = models.Model(inputs, outputs, name="bilstm_regressor")
    model.compile(optimizer=optimizers.Adam(learning_rate=learning_rate), loss="mse")
    return model


def build_transformer_model(
    sequence_length: int,
    n_features: int,
    d_model: int = 64,
    n_heads: int = 4,
    ff_dim: int = 128,
    n_layers: int = 1,
    dropout: float = 0.1,
    learning_rate: float = 1e-3,
):
    """Build a small encoder-only Transformer with a learned positional embedding."""
    import tensorflow as tf
    from tensorflow.keras import layers, models, optimizers

    inputs = layers.Input(shape=(sequence_length, n_features), name="input")
    # Project to d_model and add learned positional embedding.
    x = layers.Dense(d_model)(inputs)
    pos_idx = tf.range(start=0, limit=sequence_length, delta=1)
    pos_emb = layers.Embedding(input_dim=sequence_length, output_dim=d_model)(pos_idx)
    x = x + pos_emb  # (None, L, d_model)

    for _ in range(n_layers):
        # Self-attention block (post-LN simplification).
        attn = layers.MultiHeadAttention(
            num_heads=n_heads, key_dim=d_model // n_heads
        )(x, x)
        attn = layers.Dropout(dropout)(attn)
        x = layers.LayerNormalization(epsilon=1e-6)(x + attn)
        ff = layers.Dense(ff_dim, activation="relu")(x)
        ff = layers.Dense(d_model)(ff)
        ff = layers.Dropout(dropout)(ff)
        x = layers.LayerNormalization(epsilon=1e-6)(x + ff)

    x = layers.GlobalAveragePooling1D()(x)
    x = layers.Dense(d_model, activation="relu")(x)
    x = layers.Dropout(dropout)(x)
    outputs = layers.Dense(1, name="y")(x)
    model = models.Model(inputs, outputs, name="transformer_regressor")
    model.compile(optimizer=optimizers.Adam(learning_rate=learning_rate), loss="mse")
    return model

In [19]:
# --- 3.5 Per-target hyper-parameter search (tabular models) -----------------------
# NOTE: legacy helper, NOT used for the published results. The hyper-parameters in best_params/
# were produced by the HalvingRandomSearchCV search in §3.7 (GroupKFold(3) over site_id).
from sklearn.model_selection import RandomizedSearchCV, GroupKFold
from sklearn.metrics import make_scorer, mean_squared_error

NEG_RMSE_SCORER = make_scorer(
    lambda y_true, y_pred: -float(np.sqrt(mean_squared_error(y_true, y_pred))),
    greater_is_better=True,
)

BEST_PARAMS_DIR = RUN_DIR / "best_params"
BEST_PARAMS_DIR.mkdir(parents=True, exist_ok=True)


def tune_tabular(
    name: str,
    pipeline_factory: Callable[[], Pipeline],
    param_distributions: dict,
    X: pd.DataFrame,
    y: np.ndarray,
    groups: np.ndarray,
    target: str,
    n_iter: int | None = None,
    cv_splits: int | None = None,
) -> dict:
    """Run a cave-aware RandomizedSearchCV and persist the best parameters."""
    n_iter = int(n_iter or CONFIG.n_random_search_iter)
    cv_splits = int(cv_splits or CONFIG.inner_cv_groupkfold_splits)
    if CONFIG.fast_mode:
        n_iter = max(3, n_iter // 6)
        cv_splits = 3

    cv = GroupKFold(n_splits=cv_splits)
    pipe = pipeline_factory()

    log.info("[%s | %-22s] tuning: n_iter=%d, GroupKFold(%d) over site_id",
             name, target, n_iter, cv_splits)
    search = RandomizedSearchCV(
        estimator=pipe,
        param_distributions=param_distributions,
        n_iter=n_iter,
        cv=cv,
        scoring=NEG_RMSE_SCORER,
        n_jobs=-1,
        random_state=CONFIG.random_seed,
        refit=False,
        verbose=0,
    )
    search.fit(X, y, groups=groups)

    best = {
        "model": name,
        "target": target,
        "best_neg_rmse": float(search.best_score_),
        "best_rmse": -float(search.best_score_),
        "best_params": {k: (v.item() if hasattr(v, "item") else v)
                        for k, v in search.best_params_.items()},
        "n_iter": n_iter,
        "cv_splits": cv_splits,
    }
    out_path = BEST_PARAMS_DIR / f"{name}_{target}.json"
    out_path.write_text(json.dumps(best, indent=2, default=str))
    log.info("[%s | %-22s] best RMSE=%.4f (params -> %s)",
             name, target, best["best_rmse"], out_path.name)
    return best

In [20]:
# --- 3.6 Per-target hyper-parameter search (DL sequence models) -------------------
def _eval_dl_candidate(
    factory: Callable[[int, int], Any],
    X_full: pd.DataFrame,
    y_full: np.ndarray,
    site_ids: np.ndarray,
    entity_ids: np.ndarray,
    cv_splits: int,
    seq_len: int,
    n_features_after_pre: int,
) -> float:
    """Mean RMSE across cave-aware GroupKFold folds for one DL hyper-param candidate."""
    cv = GroupKFold(n_splits=cv_splits)
    rmses: list[float] = []
    for tr, te in cv.split(X_full, groups=site_ids):
        pre = make_preprocessor()
        Xtr = pre.fit_transform(X_full.iloc[tr])
        Xte = pre.transform(X_full.iloc[te])
        # If preprocessor's CorrelationDropper changed the column count vs.
        # n_features_after_pre, rebuild the wrapper to match.
        wrapper = SequenceWrapper(
            model_factory=factory,
            sequence_length=seq_len,
            n_features=Xtr.shape[1],
            batch_size=CONFIG.dl_batch_size,
            max_epochs=CONFIG.dl_max_epochs if not CONFIG.fast_mode else 8,
            early_stop_patience=CONFIG.dl_early_stop_patience,
            verbose=0,
        )
        wrapper.fit(Xtr, y_full[tr], entity_ids=entity_ids[tr])
        y_hat = wrapper.predict(Xte, entity_ids=entity_ids[te])
        rmses.append(float(np.sqrt(mean_squared_error(y_full[te], y_hat))))
    return float(np.mean(rmses))


def tune_dl(
    name: str,
    candidate_factories: list[tuple[dict, Callable[[int, int], Any]]],
    X: pd.DataFrame,
    y: np.ndarray,
    site_ids: np.ndarray,
    entity_ids: np.ndarray,
    target: str,
    n_iter: int | None = None,
    cv_splits: int = 3,
) -> dict:
    """Random search over a list of (params, factory) candidates with cave-aware CV."""
    n_iter = int(n_iter or CONFIG.dl_n_random_search_iter)
    if CONFIG.fast_mode:
        n_iter = max(2, n_iter // 4)

    rng = np.random.default_rng(CONFIG.random_seed)
    if len(candidate_factories) > n_iter:
        chosen_idx = rng.choice(len(candidate_factories), size=n_iter, replace=False)
    else:
        chosen_idx = np.arange(len(candidate_factories))

    log.info("[%s | %-22s] DL tuning: %d candidates, GroupKFold(%d) over site_id",
             name, target, len(chosen_idx), cv_splits)

    best = {"rmse": np.inf, "params": None}
    # Get the post-preprocess column count from a one-shot fit to size the wrapper.
    pre_demo = make_preprocessor()
    n_features_after_pre = pre_demo.fit_transform(X).shape[1]

    seq_len = CONFIG.sequence_length
    history = []
    for k, idx in enumerate(chosen_idx):
        params, factory = candidate_factories[int(idx)]
        try:
            mean_rmse = _eval_dl_candidate(
                factory=factory,
                X_full=X, y_full=y,
                site_ids=site_ids, entity_ids=entity_ids,
                cv_splits=cv_splits,
                seq_len=seq_len,
                n_features_after_pre=n_features_after_pre,
            )
        except Exception as e:  # noqa: BLE001
            log.warning("[%s] candidate %d failed: %s", name, k, e)
            continue
        log.info("  cand %2d/%d  RMSE=%.4f  params=%s", k + 1, len(chosen_idx), mean_rmse, params)
        history.append({"k": k, "params": params, "rmse": mean_rmse})
        if mean_rmse < best["rmse"]:
            best = {"rmse": mean_rmse, "params": params, "factory_index": int(idx)}

    out = {
        "model": name, "target": target,
        "best_rmse": float(best["rmse"]),
        "best_params": best["params"],
        "history": history,
    }
    (BEST_PARAMS_DIR / f"{name}_{target}.json").write_text(json.dumps(out, indent=2, default=str))
    log.info("[%s | %-22s] DL best RMSE=%.4f params=%s", name, target, best["rmse"], best["params"])
    return out


def make_bilstm_candidates() -> list[tuple[dict, Callable[[int, int], Any]]]:
    candidates = []
    for u in CONFIG.bilstm_units_grid:
        for nl in CONFIG.bilstm_layers_grid:
            for dr in CONFIG.bilstm_dropout_grid:
                params = {"units": int(u), "n_layers": int(nl), "dropout": float(dr)}
                factory = (lambda L, F, p=params: build_bilstm_model(L, F, **p))
                candidates.append((params, factory))
    return candidates


def make_transformer_candidates() -> list[tuple[dict, Callable[[int, int], Any]]]:
    candidates = []
    for d in CONFIG.transformer_dmodel_grid:
        for h in CONFIG.transformer_heads_grid:
            for ff in CONFIG.transformer_ff_grid:
                for nl in CONFIG.transformer_layers_grid:
                    if d % h != 0:
                        continue
                    params = {"d_model": int(d), "n_heads": int(h),
                              "ff_dim": int(ff), "n_layers": int(nl)}
                    factory = (lambda L, F, p=params: build_transformer_model(L, F, **p))
                    candidates.append((params, factory))
    return candidates

In [21]:
# --- 3.7 (PART 2) Reload best_params written by PART 1 ----------------------------
# In PART 1 every (model, target) pair's best hyper-parameters were saved to
#   artifacts/<run-id>/best_params/<model>_<target>.json
# PART 2 simply reads them back and rebuilds the in-memory dict the LOCO,
# permutation-test and within-cave loops need.

import xgboost as xgb            # ensure tabular factories' deps are importable here
from sklearn.ensemble import RandomForestRegressor

try:
    from catboost import CatBoostRegressor
    HAS_CATBOOST = True
except ImportError:
    log.warning("catboost is not installed; CatBoost rows will be skipped in PART 2.")
    HAS_CATBOOST = False

try:
    import tensorflow as tf  # noqa: F401
    HAS_TF = True
except Exception as e:  # noqa: BLE001
    log.warning("TensorFlow not available (%s) — DL evaluation disabled in PART 2", e)
    HAS_TF = False

BEST_PARAMS_DIR = RUN_DIR / "best_params"
ALL_MODELS_TABULAR = ["RandomForest", "XGBoost"] + (["CatBoost"] if HAS_CATBOOST else [])
# DL is only meaningful in PART 2 if PART 1 also produced best_params for it.
ALL_MODELS_DL = []
if HAS_TF:
    for _name in ("BiLSTM", "Transformer"):
        if any(BEST_PARAMS_DIR.glob(f"{_name}_*.json")):
            ALL_MODELS_DL.append(_name)

best_params_all: dict[tuple[str, str], dict] = {}
for tgt in CONFIG.targets:
    for name in ALL_MODELS_TABULAR + ALL_MODELS_DL:
        p = BEST_PARAMS_DIR / f"{name}_{tgt}.json"
        if p.exists():
            best_params_all[(name, tgt)] = json.loads(p.read_text())
        else:
            log.warning("best_params missing for (%s, %s) at %s — skipping this pair", name, tgt, p)

log.info("Loaded best_params for %d (model, target) pairs from %s",
         len(best_params_all), BEST_PARAMS_DIR)

---

## 4. LOCO evaluation loop, permutation tests, and paired-statistical tests

This is the **headline experiment** of the paper (Tables 1–3). For every `(model, target)` pair we:

1. **Loop over LOCO folds.** Each fold holds out one entire cave; the model is fit on the other 40 caves' samples (or the other 40's per-entity sequences for the DL models) using the per-target best hyper-parameters from §3.7.
2. **Score** with R², MAE, RMSE, MdAE, KS-D, and a calibration slope/intercept (regress observed on predicted).
3. **Aggregate per-fold metrics** with bootstrap 95 % CIs (`CONFIG.bootstrap_resamples=1000`).
4. **Run an honest permutation test**: refit the entire pipeline on shuffled `y_train` (`CONFIG.permutation_n_iter=200` re-fits) so the null distribution captures everything the model could ever learn from spurious feature–target alignments.
5. **Pairwise paired Wilcoxon** between every pair of models on per-fold RMSEs, with Holm–Bonferroni correction across the `5 choose 2 = 10` comparisons per target.

Per-fold predictions go to `artifacts/<run-id>/predictions/<model>_<target>.parquet`; aggregated metrics go to `artifacts/<run-id>/metrics/loco_metrics.csv`.

In [22]:
# --- 4.1 Metrics module -----------------------------------------------------------
from scipy import stats as scipy_stats
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
)


def _safe(fn):
    """Return NaN instead of raising if the inputs are degenerate."""
    def wrapped(y_true, y_pred):
        try:
            return float(fn(y_true, y_pred))
        except Exception:
            return float("nan")
    return wrapped


@_safe
def _r2(y, p): return r2_score(y, p)
@_safe
def _mae(y, p): return mean_absolute_error(y, p)
@_safe
def _rmse(y, p): return float(np.sqrt(mean_squared_error(y, p)))
@_safe
def _mdae(y, p): return float(np.median(np.abs(np.asarray(y) - np.asarray(p))))
@_safe
def _ks_d(y, p): return float(scipy_stats.ks_2samp(np.asarray(y), np.asarray(p)).statistic)


def _calibration(y_true, y_pred):
    """Regress observed = a + b * predicted; return (slope, intercept).

    A perfectly calibrated model has slope=1, intercept=0.
    """
    y = np.asarray(y_true, dtype=float)
    p = np.asarray(y_pred, dtype=float)

    mask = np.isfinite(y) & np.isfinite(p)
    y = y[mask]
    p = p[mask]

    # Degenerate fold: not enough points or no variance in predictor/target.
    if len(y) < 3 or np.std(p) == 0 or np.std(y) == 0:
        return float("nan"), float("nan")

    try:
        slope, intercept, *_ = scipy_stats.linregress(p, y)
        return float(slope), float(intercept)
    except Exception:
        return float("nan"), float("nan")


def compute_metrics(y_true: np.ndarray, y_pred: np.ndarray) -> dict[str, float]:
    """Compute every reported metric for one (y_true, y_pred) pair."""
    slope, intercept = _calibration(y_true, y_pred)
    return {
        "n": int(len(y_true)),
        "r2": _r2(y_true, y_pred),
        "mae": _mae(y_true, y_pred),
        "rmse": _rmse(y_true, y_pred),
        "mdae": _mdae(y_true, y_pred),
        "ks_d": _ks_d(y_true, y_pred),
        "calibration_slope": slope,
        "calibration_intercept": intercept,
    }


def bootstrap_ci(values: np.ndarray, n_resamples: int = None, alpha: float = 0.05,
                 random_state: int | None = None) -> tuple[float, float, float]:
    """Percentile bootstrap mean + (lo, hi) CI."""
    n_resamples = int(n_resamples or CONFIG.bootstrap_resamples)
    rng = np.random.default_rng(CONFIG.random_seed if random_state is None else random_state)
    vals = np.asarray(values, dtype=float)
    vals = vals[~np.isnan(vals)]
    if len(vals) < 2:
        return float(np.nanmean(vals) if len(vals) else np.nan), float("nan"), float("nan")
    boots = rng.choice(vals, size=(n_resamples, len(vals)), replace=True).mean(axis=1)
    lo, hi = np.percentile(boots, [100 * alpha / 2, 100 * (1 - alpha / 2)])
    return float(vals.mean()), float(lo), float(hi)

In [23]:
# --- 4.2 LOCO evaluation loop (tabular & DL share the same outer skeleton) --------
PRED_DIR = RUN_DIR / "predictions"
METRICS_DIR = RUN_DIR / "metrics"
PRED_DIR.mkdir(parents=True, exist_ok=True)
METRICS_DIR.mkdir(parents=True, exist_ok=True)


def _build_tabular_with_best(name: str, best: dict) -> Pipeline:
    """Reconstruct a tabular pipeline with the best-found hyper-parameters baked in."""
    if name == "RandomForest":
        pipe = make_rf_pipeline()
    elif name == "XGBoost":
        pipe = make_xgb_pipeline()
    elif name == "CatBoost":
        pipe = make_catboost_pipeline()
    else:
        raise ValueError(f"Unknown tabular model: {name}")
    if best.get("best_params"):
        pipe.set_params(**best["best_params"])
    return pipe


def _build_dl_with_best(name: str, best: dict) -> Callable[[int, int], Any]:
    """Reconstruct a Keras model factory with the best-found hyper-parameters baked in."""
    params = best.get("best_params", {})
    if name == "BiLSTM":
        return lambda L, F, p=params: build_bilstm_model(L, F, **p)
    if name == "Transformer":
        return lambda L, F, p=params: build_transformer_model(L, F, **p)
    raise ValueError(f"Unknown DL model: {name}")


def _loco_cache_paths(name: str, target: str) -> tuple[Path, Path]:
    return (PRED_DIR / f"{name}_{target}.parquet",
            METRICS_DIR / f"{name}_{target}_per_fold.csv")


def _try_load_loco_cache(name: str, target: str) -> tuple[pd.DataFrame, pd.DataFrame] | None:
    """If a previous LOCO run for (name, target) wrote both outputs, load them.

    This is the resumability hook: if Colab disconnects mid-loop, re-running
    the notebook will skip every (model, target) pair that already has both
    its predictions parquet and its per-fold metrics csv on disk.
    """
    pred_path, fold_path = _loco_cache_paths(name, target)
    if pred_path.exists() and fold_path.exists():
        try:
            pred_df = pd.read_parquet(pred_path)
            fold_df = pd.read_csv(fold_path)
            if not pred_df.empty and not fold_df.empty:
                log.info("[%s | %s] LOCO cache hit -> reusing %s + %s",
                         name, target, pred_path.name, fold_path.name)
                return pred_df, fold_df
        except Exception as e:  # noqa: BLE001
            log.warning("Cache for (%s, %s) is unreadable (%s); will recompute", name, target, e)
    return None


def run_loco_tabular(
    name: str,
    target: str,
    X: pd.DataFrame,
    y: np.ndarray,
    groups_site: np.ndarray,
    sample_ids: np.ndarray,
    site_names: np.ndarray,
    best: dict,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Run LOCO with a tabular model. Returns (predictions_df, per_fold_metrics_df).

    Cached on disk per (name, target): if both the predictions parquet and the
    per-fold metrics csv already exist, they are simply re-loaded and returned
    without re-fitting any models. This makes PART 1 of the pipeline safe to
    resume after a Colab disconnect — just re-run the notebook.
    """
    cached = _try_load_loco_cache(name, target)
    if cached is not None:
        return cached

    splitter = LeaveOneCaveOut()
    pred_records: list[dict] = []
    fold_records: list[dict] = []

    for k, (tr, te) in enumerate(splitter.split(X, groups=groups_site)):
        held_out = int(groups_site[te[0]])
        held_out_name = str(site_names[te[0]])
        pipe = _build_tabular_with_best(name, best)
        pipe.fit(X.iloc[tr], y[tr])
        y_hat = pipe.predict(X.iloc[te])

        for ix, yi, yh in zip(te, y[te], y_hat):
            pred_records.append({
                "fold": k,
                "site_id": held_out,
                "site_name": held_out_name,
                "sample_id": int(sample_ids[ix]),
                "target": target,
                "model": name,
                "y_true": float(yi),
                "y_pred": float(yh),
            })

        m = compute_metrics(y[te], y_hat)
        m.update({"fold": k, "model": name, "target": target,
                  "site_id": held_out, "site_name": held_out_name})
        fold_records.append(m)

    pred_df = pd.DataFrame(pred_records)
    fold_df = pd.DataFrame(fold_records)
    pred_df.to_parquet(PRED_DIR / f"{name}_{target}.parquet", index=False)
    fold_df.to_csv(METRICS_DIR / f"{name}_{target}_per_fold.csv", index=False)
    log.info("LOCO done [%s | %s]: 41 folds, mean RMSE=%.4f, mean R²=%.4f",
             name, target,
             float(fold_df["rmse"].mean()), float(fold_df["r2"].mean()))
    return pred_df, fold_df


def run_loco_dl(
    name: str,
    target: str,
    X: pd.DataFrame,
    y: np.ndarray,
    groups_site: np.ndarray,
    groups_entity: np.ndarray,
    sample_ids: np.ndarray,
    site_names: np.ndarray,
    best: dict,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Run LOCO with a DL sequence model.

    Same per-(name, target) caching as `run_loco_tabular`: if previous outputs
    exist on disk, they are reused so the notebook is resumable after a Colab
    disconnect.
    """
    cached = _try_load_loco_cache(name, target)
    if cached is not None:
        return cached

    splitter = LeaveOneCaveOut()
    pred_records: list[dict] = []
    fold_records: list[dict] = []
    factory = _build_dl_with_best(name, best)

    for k, (tr, te) in enumerate(splitter.split(X, groups=groups_site)):
        held_out = int(groups_site[te[0]])
        held_out_name = str(site_names[te[0]])

        pre = make_preprocessor()
        Xtr = pre.fit_transform(X.iloc[tr])
        Xte = pre.transform(X.iloc[te])

        wrapper = SequenceWrapper(
            model_factory=factory,
            sequence_length=CONFIG.sequence_length,
            n_features=Xtr.shape[1],
            batch_size=CONFIG.dl_batch_size,
            max_epochs=CONFIG.dl_max_epochs if not CONFIG.fast_mode else 8,
            early_stop_patience=CONFIG.dl_early_stop_patience,
            verbose=0,
        )
        wrapper.fit(Xtr, y[tr], entity_ids=groups_entity[tr])
        y_hat = wrapper.predict(Xte, entity_ids=groups_entity[te])

        for ix, yi, yh in zip(te, y[te], y_hat):
            pred_records.append({
                "fold": k,
                "site_id": held_out,
                "site_name": held_out_name,
                "sample_id": int(sample_ids[ix]),
                "target": target,
                "model": name,
                "y_true": float(yi),
                "y_pred": float(yh),
            })

        m = compute_metrics(y[te], y_hat)
        m.update({"fold": k, "model": name, "target": target,
                  "site_id": held_out, "site_name": held_out_name})
        fold_records.append(m)

    pred_df = pd.DataFrame(pred_records)
    fold_df = pd.DataFrame(fold_records)
    pred_df.to_parquet(PRED_DIR / f"{name}_{target}.parquet", index=False)
    fold_df.to_csv(METRICS_DIR / f"{name}_{target}_per_fold.csv", index=False)
    log.info("LOCO done [%s | %s]: 41 folds, mean RMSE=%.4f, mean R²=%.4f",
             name, target,
             float(fold_df["rmse"].mean()), float(fold_df["r2"].mean()))
    return pred_df, fold_df

In [24]:
# --- 4.3 (PART 2) Reload LOCO predictions and per-fold metrics --------------------
# PART 1 wrote per-(model, target) outputs to:
#   artifacts/<run-id>/predictions/<model>_<target>.parquet
#   artifacts/<run-id>/metrics/<model>_<target>_per_fold.csv
# We reload them all into the canonical in-memory dicts so every downstream
# cell (plotting, perm tests, pairwise Wilcoxon, summary tables) just works.

# PRED_DIR / METRICS_DIR are already created by cell 4.2 above; just sanity-check
# that PART 1 actually populated them.
loco_files = list(PRED_DIR.glob("*.parquet")) if PRED_DIR.exists() else []
if not loco_files:
    raise FileNotFoundError(
        f"No LOCO predictions found under {PRED_DIR}.\n"
        f"Did PART 1 finish on this RUN_ID ({RUN_TS})?\n"
        f"Look for {RUN_DIR / 'PART1_COMPLETE.txt'} — it is written at the end "
        f"of PART 1's section 4.3.\n"
        f"If PART 1 is still in progress, wait for it to finish and re-run this notebook."
    )

loco_predictions: dict[tuple[str, str], pd.DataFrame] = {}
loco_per_fold:    dict[tuple[str, str], pd.DataFrame] = {}

for tgt in CONFIG.targets:
    for name in ALL_MODELS_TABULAR + ALL_MODELS_DL:
        pred_path = PRED_DIR / f"{name}_{tgt}.parquet"
        fold_path = METRICS_DIR / f"{name}_{tgt}_per_fold.csv"
        if pred_path.exists() and fold_path.exists():
            loco_predictions[(name, tgt)] = pd.read_parquet(pred_path)
            loco_per_fold[(name, tgt)] = pd.read_csv(fold_path)
        else:
            log.warning("LOCO outputs missing for (%s, %s) — that model/target pair will not appear in plots.",
                        name, tgt)

log.info("Loaded LOCO predictions/per-fold metrics for %d (model, target) pairs",
         len(loco_per_fold))

In [25]:
# --- 4.4 Aggregate metrics for publication + diagnostics ---------------------------
# IMPORTANT:
# In cave-wise LOCO, the arithmetic mean of per-fold R2 can look overly negative
# because small/low-variance held-out caves can produce extreme R2 values.
# For headline reporting, we therefore use pooled out-of-fold (OOF) metrics,
# computed on all held-out predictions concatenated together.

# (A) Keep fold-level aggregates (with bootstrap CIs) for diagnostics/supplement.
agg_records: list[dict] = []
for (name, tgt), fold_df in loco_per_fold.items():
    rec = {"model": name, "target": tgt, "n_folds": len(fold_df)}
    for col in ["r2", "mae", "rmse", "mdae", "ks_d",
                "calibration_slope", "calibration_intercept"]:
        mean, lo, hi = bootstrap_ci(fold_df[col].values)
        rec[f"{col}_mean"] = mean
        rec[f"{col}_ci_lo"] = lo
        rec[f"{col}_ci_hi"] = hi
    agg_records.append(rec)

loco_metrics = pd.DataFrame(agg_records).sort_values(["target", "rmse_mean"])
loco_metrics.to_csv(METRICS_DIR / "loco_metrics.csv", index=False)
log.info("LOCO fold-aggregate metrics -> %s", METRICS_DIR / "loco_metrics.csv")

# (B) Publication headline table: pooled OOF metrics per (model, target).
pooled_records: list[dict] = []
for (name, tgt), pred_df in loco_predictions.items():
    y_true = pred_df["y_true"].to_numpy(dtype=float)
    y_pred = pred_df["y_pred"].to_numpy(dtype=float)

    fold_df = loco_per_fold[(name, tgt)]
    if "n" in fold_df.columns and fold_df["n"].notna().any():
        w = fold_df["n"].to_numpy(dtype=float)
    else:
        w = np.ones(len(fold_df), dtype=float)

    pooled_records.append({
        "model": name,
        "target": tgt,
        "n_rows_oof": int(len(y_true)),
        "pooled_r2": _r2(y_true, y_pred),
        "pooled_rmse": _rmse(y_true, y_pred),
        "pooled_mae": _mae(y_true, y_pred),
        "pooled_mdae": _mdae(y_true, y_pred),
        # keep these for transparency, but not headline ranking
        "fold_r2_mean_unweighted": float(np.nanmean(fold_df["r2"].values)),
        "fold_r2_mean_weighted": float(np.average(fold_df["r2"].values, weights=w)),
        "fold_r2_median": float(np.nanmedian(fold_df["r2"].values)),
    })

loco_metrics_pooled = pd.DataFrame(pooled_records).sort_values(["target", "pooled_rmse"])
loco_metrics_pooled.to_csv(METRICS_DIR / "loco_metrics_pooled.csv", index=False)
log.info("LOCO pooled OOF metrics -> %s", METRICS_DIR / "loco_metrics_pooled.csv")

print("\nLOCO PUBLICATION TABLE (pooled out-of-fold metrics; ranked by pooled RMSE):")
pub_cols = [
    "model", "target", "n_rows_oof",
    "pooled_rmse", "pooled_mae", "pooled_r2", "pooled_mdae"
]
print(loco_metrics_pooled[pub_cols].to_string(index=False))

print("\nLOCO DIAGNOSTIC TABLE (fold-level R2 summaries; interpret with caution under LOCO):")
diag_cols = [
    "model", "target",
    "fold_r2_mean_unweighted", "fold_r2_mean_weighted", "fold_r2_median"
]
print(loco_metrics_pooled[diag_cols].to_string(index=False))


LOCO PUBLICATION TABLE (pooled out-of-fold metrics; ranked by pooled RMSE):
       model            target  n_rows_oof  pooled_rmse  pooled_mae  pooled_r2  pooled_mdae
    CatBoost Mg_Ca_measurement       18004    11.743953    9.744184  -0.553329     9.419457
RandomForest Mg_Ca_measurement       18004    12.079158   10.099892  -0.643267     8.851128
     XGBoost Mg_Ca_measurement       18004    13.362438   11.034494  -1.010973     9.797978
RandomForest Sr_Ca_measurement       18004     0.327571    0.150123   0.563560     0.032695
     XGBoost Sr_Ca_measurement       18004     0.398980    0.192465   0.352534     0.046151
    CatBoost Sr_Ca_measurement       18004     0.422192    0.187217   0.275005     0.037267
    CatBoost  d13C_measurement       18004     2.816819    2.359473   0.181639     2.238964
     XGBoost  d13C_measurement       18004     3.061098    2.557223   0.033545     2.433985
RandomForest  d13C_measurement       18004     3.315789    2.764145  -0.133968     2.605795
Ran

In [26]:
# --- 4.5 Honest permutation test --------------------------------------------------
def honest_permutation_test_tabular(
    name: str,
    target: str,
    X: pd.DataFrame,
    y: np.ndarray,
    groups_site: np.ndarray,
    best: dict,
    observed_rmse: float,
    n_iter: int | None = None,
) -> dict:
    """Refit the pipeline on shuffled y_train, M times, under LOCO.

    The null distribution is the *cross-cave RMSE* a model achieves when its
    inputs are intact but the targets carry no information beyond their
    marginal distribution. p-value = fraction of permuted runs whose mean RMSE
    is ≤ the observed mean RMSE.
    """
    n_iter = int(n_iter or CONFIG.permutation_n_iter)
    if CONFIG.fast_mode:
        n_iter = max(20, n_iter // 5)

    splitter = LeaveOneCaveOut()
    rng = np.random.default_rng(CONFIG.random_seed)
    null_means: list[float] = []

    for it in range(n_iter):
        y_perm = rng.permutation(y).astype(float)
        fold_rmses: list[float] = []
        for k, (tr, te) in enumerate(splitter.split(X, groups=groups_site)):
            pipe = _build_tabular_with_best(name, best)
            pipe.fit(X.iloc[tr], y_perm[tr])
            y_hat = pipe.predict(X.iloc[te])
            fold_rmses.append(_rmse(y[te], y_hat))
        null_means.append(float(np.mean(fold_rmses)))
        if (it + 1) % max(1, n_iter // 10) == 0:
            log.info("  perm-test [%s|%s] %d/%d done (last null mean RMSE=%.4f)",
                     name, target, it + 1, n_iter, null_means[-1])

    null_arr = np.asarray(null_means)
    p_val = float((null_arr <= observed_rmse).mean())
    return {
        "model": name, "target": target,
        "observed_rmse": float(observed_rmse),
        "null_mean": float(null_arr.mean()),
        "null_std": float(null_arr.std(ddof=1)) if len(null_arr) > 1 else float("nan"),
        "null_lo": float(np.percentile(null_arr, 2.5)),
        "null_hi": float(np.percentile(null_arr, 97.5)),
        "p_value": p_val,
        "n_iter": int(n_iter),
    }


# Run perm tests only for tabular models (DL would multiply runtime by ~30×).
# For DL we report the same metric but cite the tabular perm test as evidence
# the *task* is non-trivial — a standard practice when DL is one slice of a
# larger panel.
#
# *** WARNING — this is the single most expensive cell in PART 2. ***
# With the defaults (permutation_only_best_per_target=True, permutation_n_iter=50),
# each (best_model, target) perm test refits the LOCO pipeline 50 × 41 ≈ 2050
# times. On Colab Pro that can easily run 6–24 h *per target*, and 4 targets
# × ~10 h ≈ multi-session. Hence: DISABLED BY DEFAULT.
#
# To enable: flip `ENABLE_PERMUTATION_TEST = True` below. Outputs are cached
# per (model, target) in `metrics/perm/`, so even if Colab disconnects you can
# re-run this cell and pick up where you left off. The downstream plotting
# cell (6.10) gracefully skips its panel if no perm tests have been computed.
ENABLE_PERMUTATION_TEST = False
PERM_DIR = METRICS_DIR / "perm"
PERM_DIR.mkdir(parents=True, exist_ok=True)

perm_records: list[dict] = []
if ENABLE_PERMUTATION_TEST:
    for tgt in CONFIG.targets:
        # Build the candidate list. By default we only run the perm test for
        # the single best-RMSE model per target — running it for every model
        # is 5× more compute (and the qualitative claim "ML beats permuted-y
        # null" is identical for any model in the panel that reaches it).
        if CONFIG.permutation_only_best_per_target:
            candidates = (
                loco_metrics[loco_metrics["target"] == tgt]
                .sort_values("rmse_mean")
                .head(1)["model"]
                .tolist()
            )
            candidates = [m for m in candidates if m in ALL_MODELS_TABULAR]
        else:
            candidates = list(ALL_MODELS_TABULAR)

        for name in candidates:
            cache_file = PERM_DIR / f"{name}_{tgt}.json"
            if cache_file.exists():
                try:
                    perm = json.loads(cache_file.read_text())
                    log.info("[%s | %s] perm-test cache hit -> %s", name, tgt, cache_file.name)
                    perm_records.append(perm)
                    continue
                except Exception as e:  # noqa: BLE001
                    log.warning("Perm cache unreadable for (%s, %s): %s; recomputing", name, tgt, e)

            obs_rmse = float(loco_per_fold[(name, tgt)]["rmse"].mean())
            perm = honest_permutation_test_tabular(
                name=name, target=tgt,
                X=X_features, y=master_table[tgt].values,
                groups_site=groups_site,
                best=best_params_all.get((name, tgt), {}),
                observed_rmse=obs_rmse,
            )
            cache_file.write_text(json.dumps(perm, indent=2, default=str))
            perm_records.append(perm)

perm_df = pd.DataFrame(perm_records)
if not perm_df.empty:
    perm_df.to_csv(METRICS_DIR / "permutation_tests.csv", index=False)
    log.info("Permutation tests -> %s", METRICS_DIR / "permutation_tests.csv")
    print("\nHONEST PERMUTATION TEST RESULTS:")
    print(perm_df[["model", "target", "observed_rmse",
                   "null_mean", "null_std", "p_value", "n_iter"]].to_string(index=False))

In [27]:
# --- 4.6 Pairwise paired Wilcoxon between models on per-fold RMSE -----------------
from itertools import combinations
from scipy.stats import wilcoxon


def holm_bonferroni(p_values: list[float]) -> list[float]:
    """Standard Holm-Bonferroni step-down adjustment, returns adjusted p-values."""
    m = len(p_values)
    order = np.argsort(p_values)
    adj = np.empty(m)
    running_max = 0.0
    for rank, idx in enumerate(order):
        adj_p = (m - rank) * p_values[idx]
        running_max = max(running_max, adj_p)
        adj[idx] = min(1.0, running_max)
    return adj.tolist()


pairwise_records: list[dict] = []
all_models_in_panel = ALL_MODELS_TABULAR + ALL_MODELS_DL

for tgt in CONFIG.targets:
    pairs = list(combinations(all_models_in_panel, 2))
    raw_p_values: list[float] = []
    raw_records: list[dict] = []
    for a, b in pairs:
        if (a, tgt) not in loco_per_fold or (b, tgt) not in loco_per_fold:
            continue
        ra = loco_per_fold[(a, tgt)].set_index("site_id")["rmse"]
        rb = loco_per_fold[(b, tgt)].set_index("site_id")["rmse"]
        common = ra.index.intersection(rb.index)
        diff = (ra.loc[common] - rb.loc[common]).values
        try:
            stat, p = wilcoxon(diff, zero_method="wilcox", alternative="two-sided")
        except ValueError:
            stat, p = float("nan"), float("nan")
        raw_records.append({
            "target": tgt, "model_A": a, "model_B": b,
            "median_diff_rmse_A_minus_B": float(np.median(diff)),
            "n_folds": int(len(common)),
            "wilcoxon_stat": float(stat) if stat == stat else float("nan"),
            "p_raw": float(p) if p == p else float("nan"),
        })
        raw_p_values.append(p if p == p else 1.0)

    if raw_records:
        adj = holm_bonferroni(raw_p_values)
        for rec, p_adj in zip(raw_records, adj):
            rec["p_holm"] = float(p_adj)
            pairwise_records.append(rec)

pairwise_df = pd.DataFrame(pairwise_records)
if not pairwise_df.empty:
    pairwise_df.to_csv(METRICS_DIR / "pairwise_model_tests.csv", index=False)
    log.info("Pairwise paired Wilcoxon -> %s", METRICS_DIR / "pairwise_model_tests.csv")
    print("\nPAIRWISE PAIRED WILCOXON (per-fold RMSE, Holm-Bonferroni adjusted):")
    print(pairwise_df.to_string(index=False))


PAIRWISE PAIRED WILCOXON (per-fold RMSE, Holm-Bonferroni adjusted):
           target      model_A  model_B  median_diff_rmse_A_minus_B  n_folds  wilcoxon_stat    p_raw   p_holm
 d18O_measurement RandomForest  XGBoost                   -0.085782       41          269.0 0.035980 0.107941
 d18O_measurement RandomForest CatBoost                    0.033657       41          372.0 0.456341 0.456341
 d18O_measurement      XGBoost CatBoost                    0.092458       41          296.0 0.082476 0.164951
 d13C_measurement RandomForest  XGBoost                    0.137519       41          249.0 0.017863 0.035726
 d13C_measurement RandomForest CatBoost                    0.344598       41          225.0 0.006939 0.020818
 d13C_measurement      XGBoost CatBoost                    0.129829       41          312.0 0.127190 0.127190
Mg_Ca_measurement RandomForest  XGBoost                    0.056770       41          407.0 0.768080 0.960111
Mg_Ca_measurement RandomForest CatBoost            

---

## 5. Within-cave gap-fill experiment (Section 3.5 / Table 4)

This is the **secondary** experiment. Unlike the LOCO loop above (which asks "can the model generalize to a *new cave*?"), the within-cave block-gap experiment asks "can the model fill a contiguous *gap* inside a cave it has already partly seen?". This is also the only setting in which classical interpolation methods are mathematically well-defined: they need adjacent age-sorted training points from the *same* entity to bridge to, so there is no gradient under LOCO.

We evaluate the same five ML models plus four classical baselines:

| Baseline | Definition |
|---|---|
| **Cubic Spline** | scipy `CubicSpline` over `interp_age` of the in-entity training rows; evaluated at the test rows' `interp_age`. |
| **Linear Interp** | scipy `interp1d(kind="linear")` with the same setup. |
| **kNN(k=5, age+depth)** | sklearn `KNeighborsRegressor` on standardized `(interp_age, depth_sample)` of the in-entity training rows. |
| **Per-cave mean** | The mean target value across that cave's training rows; constant prediction over the test block. |

All baselines are leakage-aware in the sense that they only see training rows of the same entity — but they are *not* expected to generalize across caves and do not appear in Tables 1–3.

In [28]:
# --- 5.1 Classical baselines (cubic spline, linear, kNN, per-cave mean) -----------
from scipy.interpolate import CubicSpline, interp1d
from sklearn.neighbors import KNeighborsRegressor
from sklearn.preprocessing import StandardScaler


def _interp_per_entity(
    age_train: np.ndarray, y_train: np.ndarray, age_test: np.ndarray, kind: str
) -> np.ndarray:
    """Interpolate one entity's training rows to its test ages."""
    if len(age_train) < 4:
        # Fall back to mean if too few training points for cubic spline.
        return np.full_like(age_test, fill_value=float(np.mean(y_train)) if len(y_train) else np.nan, dtype=float)
    order = np.argsort(age_train)
    a = age_train[order]
    yv = y_train[order]
    # Drop ties (CubicSpline requires strictly increasing x).
    keep = np.concatenate([[True], np.diff(a) > 0])
    a = a[keep]; yv = yv[keep]
    if len(a) < 4 and kind == "cubic":
        kind = "linear"
    if kind == "cubic":
        spline = CubicSpline(a, yv, extrapolate=True)
        return spline(age_test)
    f = interp1d(a, yv, kind="linear", fill_value="extrapolate", assume_sorted=True)
    return f(age_test)


def _within_cache_paths(name: str, target: str) -> tuple[Path, Path]:
    return (PRED_DIR / f"WITHIN_{name}_{target}.parquet",
            METRICS_DIR / f"WITHIN_{name}_{target}_per_fold.csv")


def _try_load_within_cache(name: str, target: str) -> tuple[pd.DataFrame, pd.DataFrame] | None:
    pred_path, fold_path = _within_cache_paths(name, target)
    if pred_path.exists() and fold_path.exists():
        try:
            pred_df = pd.read_parquet(pred_path)
            fold_df = pd.read_csv(fold_path)
            if not pred_df.empty and not fold_df.empty:
                log.info("[%s | %s] within-cave cache hit", name, target)
                return pred_df, fold_df
        except Exception as e:  # noqa: BLE001
            log.warning("Within-cave cache for (%s, %s) is unreadable (%s)", name, target, e)
    return None


def baseline_predict_within_cave(
    name: str,
    target: str,
    X: pd.DataFrame,
    y: np.ndarray,
    groups_entity: np.ndarray,
    splitter: BlockedWithinCaveSplitter,
    site_id_col: np.ndarray,
    site_name_col: np.ndarray,
    sample_id_col: np.ndarray,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    cached = _try_load_within_cache(name, target)
    if cached is not None:
        return cached
    pred_records: list[dict] = []
    fold_records: list[dict] = []
    age = X["interp_age"].values
    depth = X["depth_sample"].values

    for k, (tr, te) in enumerate(splitter.split(X, groups=groups_entity)):
        ent_id = int(groups_entity[te[0]])
        site_id = int(site_id_col[te[0]])
        site_name = str(site_name_col[te[0]])
        # Restrict TRAIN to the same entity as the test block — interpolation
        # is only defined within the entity. Cross-entity training rows are
        # irrelevant for the spline, even if the splitter technically yields
        # them in `tr`.
        tr_same_ent = tr[groups_entity[tr] == ent_id]
        if len(tr_same_ent) < 2:
            continue

        # Robust fallback value used when a baseline cannot score some rows
        # because age/depth are missing in this fold.
        y_tr_ent = y[tr_same_ent]
        finite_y_tr_ent = y_tr_ent[np.isfinite(y_tr_ent)]
        if len(finite_y_tr_ent) == 0:
            continue
        fallback_mean = float(np.mean(finite_y_tr_ent))

        if name == "CubicSpline":
            tr_mask = np.isfinite(age[tr_same_ent]) & np.isfinite(y[tr_same_ent])
            te_mask = np.isfinite(age[te])
            if tr_mask.sum() < 2:
                y_hat = np.full(len(te), fill_value=fallback_mean, dtype=float)
            else:
                y_hat = np.full(len(te), fill_value=fallback_mean, dtype=float)
                y_hat[te_mask] = _interp_per_entity(
                    age[tr_same_ent][tr_mask], y[tr_same_ent][tr_mask], age[te][te_mask], kind="cubic"
                )
        elif name == "LinearInterp":
            tr_mask = np.isfinite(age[tr_same_ent]) & np.isfinite(y[tr_same_ent])
            te_mask = np.isfinite(age[te])
            if tr_mask.sum() < 2:
                y_hat = np.full(len(te), fill_value=fallback_mean, dtype=float)
            else:
                y_hat = np.full(len(te), fill_value=fallback_mean, dtype=float)
                y_hat[te_mask] = _interp_per_entity(
                    age[tr_same_ent][tr_mask], y[tr_same_ent][tr_mask], age[te][te_mask], kind="linear"
                )
        elif name == "kNN_age_depth_k5":
            tr_mask = (
                np.isfinite(age[tr_same_ent])
                & np.isfinite(depth[tr_same_ent])
                & np.isfinite(y[tr_same_ent])
            )
            te_mask = np.isfinite(age[te]) & np.isfinite(depth[te])
            y_hat = np.full(len(te), fill_value=fallback_mean, dtype=float)
            if tr_mask.sum() >= 2 and te_mask.any():
                Xtr_raw = np.column_stack([age[tr_same_ent][tr_mask], depth[tr_same_ent][tr_mask]])
                Xte_raw = np.column_stack([age[te][te_mask], depth[te][te_mask]])
                sc = StandardScaler().fit(Xtr_raw)
                Xtr = sc.transform(Xtr_raw)
                Xte = sc.transform(Xte_raw)
                knn = KNeighborsRegressor(n_neighbors=min(5, int(tr_mask.sum()))).fit(Xtr, y[tr_same_ent][tr_mask])
                y_hat[te_mask] = knn.predict(Xte)
        elif name == "PerCaveMean":
            tr_same_site = tr[site_id_col[tr] == site_id]
            if len(tr_same_site) == 0:
                continue
            y_site = y[tr_same_site]
            y_site = y_site[np.isfinite(y_site)]
            if len(y_site) == 0:
                continue
            y_hat = np.full(len(te), fill_value=float(np.mean(y_site)), dtype=float)
        else:
            raise ValueError(f"Unknown baseline: {name}")

        for ix, yi, yh in zip(te, y[te], y_hat):
            pred_records.append({
                "fold": k, "site_id": site_id, "site_name": site_name,
                "entity_id": ent_id,
                "sample_id": int(sample_id_col[ix]),
                "target": target, "model": name,
                "y_true": float(yi), "y_pred": float(yh),
            })
        m = compute_metrics(y[te], y_hat)
        m.update({"fold": k, "model": name, "target": target,
                  "site_id": site_id, "site_name": site_name, "entity_id": ent_id})
        fold_records.append(m)

    pred_df = pd.DataFrame(pred_records)
    fold_df = pd.DataFrame(fold_records)
    pred_df.to_parquet(PRED_DIR / f"WITHIN_{name}_{target}.parquet", index=False)
    fold_df.to_csv(METRICS_DIR / f"WITHIN_{name}_{target}_per_fold.csv", index=False)
    return pred_df, fold_df

In [29]:
# --- 5.2 ML models inside the within-cave block-gap regime ------------------------
def run_within_cave_tabular(
    name: str, target: str,
    X: pd.DataFrame, y: np.ndarray,
    groups_entity: np.ndarray,
    splitter: BlockedWithinCaveSplitter,
    site_id_col: np.ndarray, site_name_col: np.ndarray, sample_id_col: np.ndarray,
    best: dict,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    cached = _try_load_within_cache(name, target)
    if cached is not None:
        return cached
    pred_records: list[dict] = []
    fold_records: list[dict] = []
    for k, (tr, te) in enumerate(splitter.split(X, groups=groups_entity)):
        ent_id = int(groups_entity[te[0]])
        site_id = int(site_id_col[te[0]])
        site_name = str(site_name_col[te[0]])
        pipe = _build_tabular_with_best(name, best)
        pipe.fit(X.iloc[tr], y[tr])
        y_hat = pipe.predict(X.iloc[te])
        for ix, yi, yh in zip(te, y[te], y_hat):
            pred_records.append({
                "fold": k, "site_id": site_id, "site_name": site_name, "entity_id": ent_id,
                "sample_id": int(sample_id_col[ix]),
                "target": target, "model": name,
                "y_true": float(yi), "y_pred": float(yh),
            })
        m = compute_metrics(y[te], y_hat)
        m.update({"fold": k, "model": name, "target": target,
                  "site_id": site_id, "site_name": site_name, "entity_id": ent_id})
        fold_records.append(m)
    pred_df = pd.DataFrame(pred_records)
    fold_df = pd.DataFrame(fold_records)
    pred_df.to_parquet(PRED_DIR / f"WITHIN_{name}_{target}.parquet", index=False)
    fold_df.to_csv(METRICS_DIR / f"WITHIN_{name}_{target}_per_fold.csv", index=False)
    return pred_df, fold_df


def run_within_cave_dl(
    name: str, target: str,
    X: pd.DataFrame, y: np.ndarray,
    groups_entity: np.ndarray,
    splitter: BlockedWithinCaveSplitter,
    site_id_col: np.ndarray, site_name_col: np.ndarray, sample_id_col: np.ndarray,
    best: dict,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    cached = _try_load_within_cache(name, target)
    if cached is not None:
        return cached
    pred_records: list[dict] = []
    fold_records: list[dict] = []
    factory = _build_dl_with_best(name, best)
    for k, (tr, te) in enumerate(splitter.split(X, groups=groups_entity)):
        ent_id = int(groups_entity[te[0]])
        site_id = int(site_id_col[te[0]])
        site_name = str(site_name_col[te[0]])
        pre = make_preprocessor()
        Xtr = pre.fit_transform(X.iloc[tr])
        Xte = pre.transform(X.iloc[te])
        wrapper = SequenceWrapper(
            model_factory=factory,
            sequence_length=CONFIG.sequence_length,
            n_features=Xtr.shape[1],
            batch_size=CONFIG.dl_batch_size,
            max_epochs=CONFIG.dl_max_epochs if not CONFIG.fast_mode else 8,
            early_stop_patience=CONFIG.dl_early_stop_patience,
            verbose=0,
        )
        wrapper.fit(Xtr, y[tr], entity_ids=groups_entity[tr])
        y_hat = wrapper.predict(Xte, entity_ids=groups_entity[te])
        for ix, yi, yh in zip(te, y[te], y_hat):
            pred_records.append({
                "fold": k, "site_id": site_id, "site_name": site_name, "entity_id": ent_id,
                "sample_id": int(sample_id_col[ix]),
                "target": target, "model": name,
                "y_true": float(yi), "y_pred": float(yh),
            })
        m = compute_metrics(y[te], y_hat)
        m.update({"fold": k, "model": name, "target": target,
                  "site_id": site_id, "site_name": site_name, "entity_id": ent_id})
        fold_records.append(m)
    pred_df = pd.DataFrame(pred_records)
    fold_df = pd.DataFrame(fold_records)
    pred_df.to_parquet(PRED_DIR / f"WITHIN_{name}_{target}.parquet", index=False)
    fold_df.to_csv(METRICS_DIR / f"WITHIN_{name}_{target}_per_fold.csv", index=False)
    return pred_df, fold_df

In [ ]:
# --- 5.3 Drive the within-cave block-gap experiment -------------------------------
within_splitter = BlockedWithinCaveSplitter(
    block_fraction=CONFIG.block_gap_fraction,
    min_block_samples=CONFIG.block_gap_min_samples,
    min_entity_length=CONFIG.min_entity_length_for_blockgap,
    random_state=CONFIG.random_seed,
    fixed_blocks=WCBG_BLOCKS_CSV,
    sample_ids=master_table["sample_id"].values,
)

BASELINE_NAMES = ["CubicSpline", "LinearInterp", "kNN_age_depth_k5", "PerCaveMean"]

within_per_fold: dict[tuple[str, str], pd.DataFrame] = {}
within_predictions: dict[tuple[str, str], pd.DataFrame] = {}

# Optional execution controls for faster Colab recovery runs.
# - WITHIN_TARGETS: subset of targets to run (None -> all 4).
# - WITHIN_TABULAR_MODELS: subset of tabular models.
# - WITHIN_BASELINE_NAMES: subset of interpolation baselines.
# - WITHIN_FAST_MODE: lightweight within-cave profile (fewer trees/iterations).
WITHIN_TARGETS: list[str] | None = None
WITHIN_TABULAR_MODELS: list[str] | None = None
WITHIN_BASELINE_NAMES: list[str] | None = None
WITHIN_FAST_MODE: bool = False

targets_to_run = WITHIN_TARGETS if WITHIN_TARGETS is not None else list(CONFIG.targets)
tabular_to_run = WITHIN_TABULAR_MODELS if WITHIN_TABULAR_MODELS is not None else list(ALL_MODELS_TABULAR)
baselines_to_run = WITHIN_BASELINE_NAMES if WITHIN_BASELINE_NAMES is not None else list(BASELINE_NAMES)

for _t in targets_to_run:
    if _t not in CONFIG.targets:
        raise ValueError(f"Unknown target in WITHIN_TARGETS: {_t}")
for _m in tabular_to_run:
    if _m not in ALL_MODELS_TABULAR:
        raise ValueError(f"Unknown model in WITHIN_TABULAR_MODELS: {_m}")
for _b in baselines_to_run:
    if _b not in BASELINE_NAMES:
        raise ValueError(f"Unknown baseline in WITHIN_BASELINE_NAMES: {_b}")

def _with_fast_within_params(name: str, best: dict) -> dict:
    """Return adjusted best-params for a faster within-cave pass when requested."""
    if not WITHIN_FAST_MODE:
        return best
    b = dict(best) if isinstance(best, dict) else {}
    p = dict(b.get("best_params", {}) or {})
    if name == "RandomForest" and "estimator__n_estimators" in p:
        p["estimator__n_estimators"] = int(min(250, max(80, p["estimator__n_estimators"])))
    elif name == "XGBoost" and "estimator__n_estimators" in p:
        p["estimator__n_estimators"] = int(min(350, max(120, p["estimator__n_estimators"])))
    elif name == "CatBoost" and "estimator__iterations" in p:
        p["estimator__iterations"] = int(min(400, max(150, p["estimator__iterations"])))
    b["best_params"] = p
    return b

total_jobs = len(targets_to_run) * (len(baselines_to_run) + len(tabular_to_run) + len(ALL_MODELS_DL))
done_jobs = 0
log.info(
    "[WITHIN] starting: targets=%s baselines=%s tabular=%s dl=%s fast_mode=%s total_jobs=%d",
    targets_to_run, baselines_to_run, tabular_to_run, ALL_MODELS_DL, WITHIN_FAST_MODE, total_jobs,
)

for tgt in targets_to_run:
    y_tgt = master_table[tgt].values
    # Baselines first (cheap, well-defined).
    for name in baselines_to_run:
        log.info("[WITHIN] (%d/%d) baseline start: target=%s model=%s", done_jobs + 1, total_jobs, tgt, name)
        pred_df, fold_df = baseline_predict_within_cave(
            name=name, target=tgt,
            X=X_features, y=y_tgt,
            groups_entity=groups_entity, splitter=within_splitter,
            site_id_col=groups_site, site_name_col=site_names, sample_id_col=sample_ids,
        )
        within_predictions[(name, tgt)] = pred_df
        within_per_fold[(name, tgt)] = fold_df
        done_jobs += 1
        log.info(
            "[WITHIN] (%d/%d) baseline done: target=%s model=%s rows=%d folds=%d",
            done_jobs, total_jobs, tgt, name, len(pred_df), len(fold_df),
        )
    # Then the ML panel.
    for name in tabular_to_run:
        best = _with_fast_within_params(name, best_params_all.get((name, tgt), {}))
        log.info("[WITHIN] (%d/%d) tabular start: target=%s model=%s", done_jobs + 1, total_jobs, tgt, name)
        pred_df, fold_df = run_within_cave_tabular(
            name=name, target=tgt,
            X=X_features, y=y_tgt,
            groups_entity=groups_entity, splitter=within_splitter,
            site_id_col=groups_site, site_name_col=site_names, sample_id_col=sample_ids,
            best=best,
        )
        within_predictions[(name, tgt)] = pred_df
        within_per_fold[(name, tgt)] = fold_df
        done_jobs += 1
        log.info(
            "[WITHIN] (%d/%d) tabular done: target=%s model=%s rows=%d folds=%d",
            done_jobs, total_jobs, tgt, name, len(pred_df), len(fold_df),
        )
    for name in ALL_MODELS_DL:
        best = best_params_all.get((name, tgt), {})
        log.info("[WITHIN] (%d/%d) dl start: target=%s model=%s", done_jobs + 1, total_jobs, tgt, name)
        pred_df, fold_df = run_within_cave_dl(
            name=name, target=tgt,
            X=X_features, y=y_tgt,
            groups_entity=groups_entity, splitter=within_splitter,
            site_id_col=groups_site, site_name_col=site_names, sample_id_col=sample_ids,
            best=best,
        )
        within_predictions[(name, tgt)] = pred_df
        within_per_fold[(name, tgt)] = fold_df
        done_jobs += 1
        log.info(
            "[WITHIN] (%d/%d) dl done: target=%s model=%s rows=%d folds=%d",
            done_jobs, total_jobs, tgt, name, len(pred_df), len(fold_df),
        )

# Aggregate within-cave metrics with bootstrap CIs.
within_records: list[dict] = []
for (name, tgt), fold_df in within_per_fold.items():
    rec = {"model": name, "target": tgt, "n_folds": len(fold_df)}
    for col in ["r2", "mae", "rmse", "mdae", "ks_d",
                "calibration_slope", "calibration_intercept"]:
        mean, lo, hi = bootstrap_ci(fold_df[col].values)
        rec[f"{col}_mean"] = mean
        rec[f"{col}_ci_lo"] = lo
        rec[f"{col}_ci_hi"] = hi
    within_records.append(rec)

within_metrics = pd.DataFrame(within_records).sort_values(["target", "rmse_mean"])
within_metrics.to_csv(METRICS_DIR / "within_cave_metrics.csv", index=False)

print("\nWITHIN-CAVE TABLE 4 (mean ± bootstrap-95%-CI):")
print(within_metrics[["model", "target", "n_folds",
                      "rmse_mean", "rmse_ci_lo", "rmse_ci_hi",
                      "r2_mean", "r2_ci_lo", "r2_ci_hi"]].to_string(index=False))

---

## 6. Visualizations (publication suite)

Every plot below is written as a vector PDF *and* a 300-dpi PNG into `artifacts/<run-id>/figures/`. Filenames follow a stable convention so the LaTeX manuscript can `\includegraphics` them by name without renaming on each rerun.

| Section | Figure | What it shows |
|---|---|---|
| 6.1 | `cohort_map.{pdf,png}` | World map of the 41 caves coloured by sample count. |
| 6.2 | `loco_summary_bar.{pdf,png}` | Per-target mean R² ± CI for every panel model under LOCO. |
| 6.3 | `loco_pred_vs_obs__<model>__<target>.{pdf,png}` | Predicted-vs-observed hexbin with 1:1 line, per `(model,target)`. |
| 6.4 | `loco_residual_kde__<target>.{pdf,png}` | Residual KDE (one curve per model) per target. |
| 6.5 | `loco_calibration__<target>.{pdf,png}` | Calibration (binned mean predicted vs binned mean observed). |
| 6.6 | `loco_spatial_error__<model>__<target>.{pdf,png}` | World map with each cave coloured by its held-out-fold MAE. |
| 6.7 | `loco_error_vs_cave_size__<target>.{pdf,png}` | Per-cave MAE vs log(n_samples-in-cave); detects small-cave failure modes. |
| 6.8 | `permutation_feature_importance__<model>__<target>.{pdf,png}` | Top-15 permutation importances on a held-out cave. |
| 6.9 | `loco_vs_within__<target>.{pdf,png}` | Side-by-side bar chart comparing LOCO vs within-cave RMSE per model. |
| 6.10 | `permutation_test_null__<model>__<target>.{pdf,png}` | Honest perm-test null distribution with the observed RMSE marked. |

In [ ]:
# --- 6.0 Plotting setup -----------------------------------------------------------
import matplotlib
matplotlib.use("Agg")  # safe in headless / Colab
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.colors import LogNorm
from matplotlib.patches import Patch

sns.set_context("paper", font_scale=1.15)
sns.set_style("whitegrid", {"grid.linestyle": "--", "grid.alpha": 0.4})

PALETTE = {
    "RandomForest": "#1f77b4",
    "XGBoost":      "#ff7f0e",
    "CatBoost":     "#2ca02c",
    "BiLSTM":       "#d62728",
    "Transformer":  "#9467bd",
    "CubicSpline":  "#8c564b",
    "LinearInterp": "#e377c2",
    "kNN_age_depth_k5": "#7f7f7f",
    "PerCaveMean":  "#bcbd22",
}

TARGET_LABELS = {
    "d18O_measurement":  r"$\delta^{18}$O",
    "d13C_measurement":  r"$\delta^{13}$C",
    "Mg_Ca_measurement": "Mg/Ca",
    "Sr_Ca_measurement": "Sr/Ca",
}


def _save_fig(fig, stem: str) -> None:
    pdf_path = FIG_DIR / f"{stem}.pdf"
    png_path = FIG_DIR / f"{stem}.png"
    fig.tight_layout()
    fig.savefig(pdf_path, bbox_inches="tight")
    fig.savefig(png_path, bbox_inches="tight", dpi=300)
    plt.close(fig)
    log.info("figure saved -> %s (.pdf, .png)", stem)

In [ ]:
# --- 6.1 Cohort map ---------------------------------------------------------------
def plot_cohort_map():
    fig, ax = plt.subplots(figsize=(11, 5.5))
    # Hand-drawn coastline-free world: just a light grey rectangle for context
    # (avoids cartopy/contextily — heavy installs we'd rather not impose).
    ax.set_xlim(-180, 180); ax.set_ylim(-90, 90)
    ax.set_facecolor("#f5f6f8")
    ax.axhline(0, color="#dddddd", linewidth=0.8)
    ax.axvline(0, color="#dddddd", linewidth=0.8)

    sizes = np.clip(np.sqrt(cohort_audit["n_samples"]) * 6, 25, 600)
    sc = ax.scatter(
        cohort_audit["longitude"], cohort_audit["latitude"],
        s=sizes, c=cohort_audit["n_samples"], cmap="viridis",
        edgecolor="black", linewidth=0.5, alpha=0.85,
        norm=LogNorm(vmin=20, vmax=cohort_audit["n_samples"].max()),
    )
    cb = plt.colorbar(sc, ax=ax, shrink=0.7, label="samples per cave (log scale)")
    ax.set_xlabel("Longitude")
    ax.set_ylabel("Latitude")
    ax.set_title(f"SISALv3 cohort: {len(cohort_audit)} caves, "
                 f"{int(cohort_audit['n_entities'].sum())} entities, "
                 f"{int(cohort_audit['n_samples'].sum())} samples")
    _save_fig(fig, "cohort_map")


plot_cohort_map()

In [ ]:
# --- 6.1b Split diagnostics (LOCO + within-cave fold sizes) ------------------------
def plot_split_diagnostics():
    fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))

    # LOCO: per-fold held-out cave size
    if "loco_audit" in globals() and isinstance(loco_audit, pd.DataFrame) and not loco_audit.empty:
        a = loco_audit.sort_values("n_test", ascending=False).reset_index(drop=True)
        axes[0].bar(np.arange(len(a)), a["n_test"], color="#264653", edgecolor="black", linewidth=0.5)
        axes[0].set_title("LOCO fold test-size distribution")
        axes[0].set_xlabel("fold rank (largest to smallest held-out cave)")
        axes[0].set_ylabel("n_test rows")
    else:
        axes[0].text(0.5, 0.5, "LOCO audit unavailable", ha="center", va="center")
        axes[0].set_axis_off()

    # Within-cave: per-fold held-out block size
    if "within_audit" in globals() and isinstance(within_audit, pd.DataFrame) and not within_audit.empty:
        b = within_audit.sort_values("n_test", ascending=False).reset_index(drop=True)
        axes[1].bar(np.arange(len(b)), b["n_test"], color="#e9c46a", edgecolor="black", linewidth=0.5)
        axes[1].set_title("Within-cave block-gap fold sizes")
        axes[1].set_xlabel("fold rank (largest to smallest held-out block)")
        axes[1].set_ylabel("n_test rows")
    else:
        axes[1].text(0.5, 0.5, "Within-cave audit unavailable", ha="center", va="center")
        axes[1].set_axis_off()

    fig.suptitle("Split diagnostics for reproducibility audit", y=1.04)
    _save_fig(fig, "split_diagnostics")


plot_split_diagnostics()

In [ ]:
# --- 6.2 LOCO summary bar (R² with bootstrap CIs) ---------------------------------
def plot_loco_summary_bar():
    fig, axes = plt.subplots(1, len(CONFIG.targets), figsize=(4.5 * len(CONFIG.targets), 4.4),
                             sharey=False)
    if len(CONFIG.targets) == 1:
        axes = [axes]

    for ax, tgt in zip(axes, CONFIG.targets):
        sub = loco_metrics[loco_metrics["target"] == tgt].sort_values("rmse_mean")
        if sub.empty:
            ax.set_visible(False); continue
        colors = [PALETTE.get(m, "#444") for m in sub["model"]]
        bars = ax.bar(sub["model"], sub["r2_mean"], color=colors,
                      edgecolor="black", linewidth=0.8)
        yerr = np.vstack([sub["r2_mean"] - sub["r2_ci_lo"],
                          sub["r2_ci_hi"] - sub["r2_mean"]])
        ax.errorbar(sub["model"], sub["r2_mean"], yerr=yerr,
                    fmt="none", ecolor="black", capsize=3, linewidth=1.0)
        ax.set_title(TARGET_LABELS.get(tgt, tgt))
        ax.set_ylabel("LOCO R² (mean ± 95% CI)" if ax is axes[0] else "")
        ax.axhline(0.0, color="grey", linestyle=":")
        ax.tick_params(axis="x", rotation=30)
    fig.suptitle("LOCO headline: per-target R² across the 5-model panel", y=1.04)
    _save_fig(fig, "loco_summary_bar")


plot_loco_summary_bar()

In [ ]:
# --- 6.3 Predicted vs observed hexbin (per model, per target) ---------------------
def plot_pred_vs_obs(predictions_dict: dict, prefix: str = "loco"):
    for (model, tgt), df in predictions_dict.items():
        if df.empty:
            continue
        fig, ax = plt.subplots(figsize=(5.0, 4.6))
        hb = ax.hexbin(df["y_pred"], df["y_true"], gridsize=40,
                       cmap="magma_r", mincnt=1, linewidths=0.0)
        cb = plt.colorbar(hb, ax=ax, shrink=0.85, label="density")
        # 1:1 line
        lo = float(min(df["y_true"].min(), df["y_pred"].min()))
        hi = float(max(df["y_true"].max(), df["y_pred"].max()))
        pad = 0.05 * (hi - lo)
        ax.plot([lo - pad, hi + pad], [lo - pad, hi + pad],
                color="cyan", linewidth=1.2, linestyle="--", label="1:1")
        ax.set_xlim(lo - pad, hi + pad); ax.set_ylim(lo - pad, hi + pad)
        ax.set_xlabel("predicted")
        ax.set_ylabel("observed")
        rmse = _rmse(df["y_true"].values, df["y_pred"].values)
        r2 = _r2(df["y_true"].values, df["y_pred"].values)
        ax.set_title(f"{model} — {TARGET_LABELS.get(tgt, tgt)}\n"
                     f"RMSE={rmse:.3f} | R²={r2:.3f} | n={len(df):,}")
        ax.legend(loc="upper left")
        _save_fig(fig, f"{prefix}_pred_vs_obs__{model}__{tgt}")


plot_pred_vs_obs(loco_predictions, prefix="loco")

In [ ]:
# --- 6.4 Residual KDE per target (one curve per model) ----------------------------
def plot_residual_kde(predictions_dict: dict, prefix: str = "loco"):
    for tgt in CONFIG.targets:
        models_here = [m for (m, t) in predictions_dict if t == tgt]
        if not models_here:
            continue
        fig, ax = plt.subplots(figsize=(7, 4.6))
        for m in sorted(models_here, key=lambda x: PALETTE.get(x, "#000000")):
            df = predictions_dict[(m, tgt)]
            if df.empty: continue
            res = (df["y_pred"] - df["y_true"]).values
            sns.kdeplot(res, ax=ax, color=PALETTE.get(m, "#444"),
                        linewidth=2.2, label=f"{m} (n={len(res):,})")
        ax.axvline(0.0, color="black", linewidth=0.8, linestyle=":")
        ax.set_xlabel(f"residual ({TARGET_LABELS.get(tgt, tgt)} predicted − observed)")
        ax.set_ylabel("density")
        ax.set_title(f"{prefix.upper()} residual distribution — {TARGET_LABELS.get(tgt, tgt)}")
        ax.legend(loc="upper right", fontsize="small")
        _save_fig(fig, f"{prefix}_residual_kde__{tgt}")


plot_residual_kde(loco_predictions, prefix="loco")

In [ ]:
# --- 6.5 Calibration curves -------------------------------------------------------
def plot_calibration(predictions_dict: dict, prefix: str = "loco", n_bins: int = 20):
    for tgt in CONFIG.targets:
        models_here = [m for (m, t) in predictions_dict if t == tgt]
        if not models_here:
            continue
        fig, ax = plt.subplots(figsize=(6, 5.5))
        for m in sorted(models_here):
            df = predictions_dict[(m, tgt)]
            if df.empty: continue
            yp = df["y_pred"].values; yt = df["y_true"].values
            bin_edges = np.linspace(np.percentile(yp, 1), np.percentile(yp, 99), n_bins + 1)
            bin_idx = np.digitize(yp, bin_edges) - 1
            bin_idx = np.clip(bin_idx, 0, n_bins - 1)
            mean_p, mean_t, sem_t = [], [], []
            for b in range(n_bins):
                mask = bin_idx == b
                if mask.sum() < 4:
                    continue
                mean_p.append(yp[mask].mean()); mean_t.append(yt[mask].mean())
                sem_t.append(yt[mask].std(ddof=1) / np.sqrt(mask.sum()))
            mean_p = np.array(mean_p); mean_t = np.array(mean_t); sem_t = np.array(sem_t)
            if len(mean_p) == 0:
                continue
            ax.errorbar(mean_p, mean_t, yerr=sem_t, color=PALETTE.get(m, "#444"),
                        marker="o", markersize=4, linewidth=1.5, capsize=3, label=m)
        # 1:1 reference
        x_lo, x_hi = ax.get_xlim()
        ax.plot([x_lo, x_hi], [x_lo, x_hi], color="black", linestyle="--",
                linewidth=1.0, label="perfect calibration")
        ax.set_xlabel("mean predicted (per bin)")
        ax.set_ylabel("mean observed (per bin)")
        ax.set_title(f"Calibration — {TARGET_LABELS.get(tgt, tgt)}")
        ax.legend(loc="upper left", fontsize="small")
        _save_fig(fig, f"{prefix}_calibration__{tgt}")


plot_calibration(loco_predictions, prefix="loco")

In [ ]:
# --- 6.6 Spatial error map: per-cave MAE -----------------------------------------
def plot_spatial_error_map(per_fold_dict: dict):
    site_geo = (
        cohort_audit.set_index("site_id")[["latitude", "longitude", "n_samples"]]
    )
    for (name, tgt), fold_df in per_fold_dict.items():
        if fold_df.empty:
            continue
        merged = fold_df.merge(
            site_geo.reset_index(), on="site_id", how="left"
        )
        fig, ax = plt.subplots(figsize=(11, 5.5))
        ax.set_xlim(-180, 180); ax.set_ylim(-90, 90)
        ax.set_facecolor("#f5f6f8")
        ax.axhline(0, color="#dddddd", linewidth=0.8)
        ax.axvline(0, color="#dddddd", linewidth=0.8)
        sc = ax.scatter(
            merged["longitude"], merged["latitude"],
            s=np.clip(np.sqrt(merged["n_samples"]) * 6, 25, 600),
            c=merged["mae"], cmap="Reds",
            edgecolor="black", linewidth=0.5, alpha=0.9,
        )
        plt.colorbar(sc, ax=ax, shrink=0.7, label="LOCO MAE on held-out cave")
        ax.set_title(f"Spatial error — {name} — {TARGET_LABELS.get(tgt, tgt)}")
        ax.set_xlabel("Longitude"); ax.set_ylabel("Latitude")
        _save_fig(fig, f"loco_spatial_error__{name}__{tgt}")


plot_spatial_error_map(loco_per_fold)

In [ ]:
# --- 6.7 LOCO error vs cave size --------------------------------------------------
def plot_error_vs_cave_size():
    site_n = cohort_audit.set_index("site_id")["n_samples"]
    for tgt in CONFIG.targets:
        fig, ax = plt.subplots(figsize=(7, 4.6))
        for (name, t), fold_df in loco_per_fold.items():
            if t != tgt or fold_df.empty: continue
            merged = fold_df.assign(n_samples=fold_df["site_id"].map(site_n))
            ax.scatter(merged["n_samples"], merged["mae"],
                       color=PALETTE.get(name, "#444"), label=name,
                       s=42, alpha=0.85, edgecolor="black", linewidth=0.4)
        ax.set_xscale("log")
        ax.set_xlabel("samples in held-out cave (log scale)")
        ax.set_ylabel("LOCO MAE")
        ax.set_title(f"Per-cave error vs cave size — {TARGET_LABELS.get(tgt, tgt)}")
        ax.legend(loc="best", fontsize="small")
        _save_fig(fig, f"loco_error_vs_cave_size__{tgt}")


plot_error_vs_cave_size()

In [ ]:
# --- 6.8 Permutation feature importance on a held-out cave ------------------------
from sklearn.inspection import permutation_importance


def plot_permutation_importance(top_k: int = 15):
    """For each (tabular-model, target) pair, compute permutation importance on
    the *held-out* fold of the largest-test cave, so the importance ranking
    reflects out-of-distribution attribution."""
    largest = loco_metrics.loc[loco_metrics["n_folds"] > 0]  # placeholder filter
    largest_site = int(cohort_audit.iloc[0]["site_id"])

    splitter = LeaveOneCaveOut()
    held_idx_by_site = {
        int(groups_site[te[0]]): (tr, te)
        for tr, te in splitter.split(X_features, groups=groups_site)
    }
    if largest_site not in held_idx_by_site:
        return
    tr, te = held_idx_by_site[largest_site]

    # Use feature names from the preprocessor's output
    pre = make_preprocessor()
    pre.fit(X_features.iloc[tr])
    feat_names = pre.named_steps["preprocess"].get_feature_names_out().tolist()

    for tgt in CONFIG.targets:
        y_tgt = master_table[tgt].values
        for name in ALL_MODELS_TABULAR:
            best = best_params_all.get((name, tgt), {})
            pipe = _build_tabular_with_best(name, best)
            pipe.fit(X_features.iloc[tr], y_tgt[tr])
            try:
                # Permutation importance on the held-out test data, using the
                # full pipeline so the permutation is over RAW features (the
                # interpretable ones) and the preprocessor is invoked inside.
                r = permutation_importance(
                    pipe, X_features.iloc[te], y_tgt[te],
                    n_repeats=10, random_state=CONFIG.random_seed,
                    scoring="neg_mean_squared_error", n_jobs=-1,
                )
            except Exception as e:  # noqa: BLE001
                log.warning("perm importance failed for %s | %s: %s", name, tgt, e)
                continue
            # Feature names here are the *raw* X_features columns
            raw_feat = X_features.columns.tolist()
            order = np.argsort(r.importances_mean)[::-1][:top_k]
            fig, ax = plt.subplots(figsize=(7, 0.36 * top_k + 1.2))
            ax.barh(
                np.arange(len(order))[::-1],
                r.importances_mean[order],
                xerr=r.importances_std[order],
                color=PALETTE.get(name, "#444"), edgecolor="black", linewidth=0.5,
            )
            ax.set_yticks(np.arange(len(order))[::-1])
            ax.set_yticklabels([raw_feat[i] for i in order])
            ax.set_xlabel("permutation importance (Δ neg-MSE)")
            ax.set_title(f"{name} — {TARGET_LABELS.get(tgt, tgt)} — held-out cave id={largest_site}")
            _save_fig(fig, f"permutation_feature_importance__{name}__{tgt}")


plot_permutation_importance(top_k=15)

In [ ]:
# --- 6.9 LOCO vs Within-cave RMSE comparison --------------------------------------
def plot_loco_vs_within():
    if not within_per_fold:
        log.warning("within_per_fold is empty; skipping LOCO-vs-within comparison")
        return
    for tgt in CONFIG.targets:
        rows = []
        for name in ALL_MODELS_TABULAR + ALL_MODELS_DL:
            loco_rmse = loco_per_fold.get((name, tgt), pd.DataFrame())["rmse"].mean() if (name, tgt) in loco_per_fold else np.nan
            within_rmse = within_per_fold.get((name, tgt), pd.DataFrame())["rmse"].mean() if (name, tgt) in within_per_fold else np.nan
            rows.append({"model": name, "regime": "LOCO",      "rmse": loco_rmse})
            rows.append({"model": name, "regime": "WithinCave", "rmse": within_rmse})
        df = pd.DataFrame(rows).dropna(subset=["rmse"])
        if df.empty:
            continue
        fig, ax = plt.subplots(figsize=(7.5, 4.6))
        sns.barplot(
            data=df, x="model", y="rmse", hue="regime", ax=ax,
            palette={"LOCO": "#264653", "WithinCave": "#e9c46a"},
            edgecolor="black", linewidth=0.7,
        )
        ax.set_ylabel("mean RMSE across folds")
        ax.set_title(f"LOCO vs within-cave RMSE — {TARGET_LABELS.get(tgt, tgt)}")
        ax.tick_params(axis="x", rotation=20)
        ax.legend(title="evaluation regime")
        _save_fig(fig, f"loco_vs_within__{tgt}")


plot_loco_vs_within()

In [ ]:
# --- 6.10 Permutation-test null distribution panel --------------------------------
def plot_permutation_null():
    if perm_df.empty:
        log.warning("perm_df is empty; skipping permutation-null plot")
        return
    for tgt in CONFIG.targets:
        sub = perm_df[perm_df["target"] == tgt]
        if sub.empty: continue
        n_models = len(sub)
        fig, axes = plt.subplots(1, n_models, figsize=(4 * n_models, 4), sharey=True)
        if n_models == 1:
            axes = [axes]
        for ax, (_, row) in zip(axes, sub.iterrows()):
            mu, sd = row["null_mean"], row["null_std"]
            xs = np.linspace(mu - 4 * sd, mu + 4 * sd, 200) if sd > 0 else np.array([mu])
            if sd > 0:
                pdf = (1.0 / (sd * np.sqrt(2 * np.pi))) * np.exp(-0.5 * ((xs - mu) / sd) ** 2)
                ax.plot(xs, pdf, color=PALETTE.get(row["model"], "#444"), linewidth=2.0,
                        label=f"null ~ N({mu:.2f}, {sd:.2f})")
                ax.fill_between(xs, pdf, alpha=0.22, color=PALETTE.get(row["model"], "#444"))
            ax.axvline(row["observed_rmse"], color="red", linestyle="--", linewidth=1.5,
                       label=f"observed RMSE={row['observed_rmse']:.3f}")
            ax.set_title(f"{row['model']}\np = {row['p_value']:.3g}")
            ax.set_xlabel("RMSE under permuted y")
            ax.set_ylabel("density" if ax is axes[0] else "")
            ax.legend(fontsize="small")
        fig.suptitle(f"Honest permutation null — {TARGET_LABELS.get(tgt, tgt)}", y=1.05)
        _save_fig(fig, f"permutation_test_null__{tgt}")


plot_permutation_null()

In [ ]:
# --- 6.11 Final summary tables for the manuscript ---------------------------------
def _format_metric_row(row, m_col, lo_col, hi_col, fmt="{:.3f}"):
    return f"{fmt.format(row[m_col])} [{fmt.format(row[lo_col])}, {fmt.format(row[hi_col])}]"


def export_summary_tables():
    # Table 1: LOCO headline R² + RMSE, per model per target.
    headline = loco_metrics.copy()
    headline["R2"] = headline.apply(lambda r: _format_metric_row(r, "r2_mean", "r2_ci_lo", "r2_ci_hi"), axis=1)
    headline["RMSE"] = headline.apply(lambda r: _format_metric_row(r, "rmse_mean", "rmse_ci_lo", "rmse_ci_hi"), axis=1)
    headline["MAE"] = headline.apply(lambda r: _format_metric_row(r, "mae_mean", "mae_ci_lo", "mae_ci_hi"), axis=1)
    out = headline[["target", "model", "n_folds", "R2", "RMSE", "MAE"]].sort_values(["target", "model"])
    out_csv = METRICS_DIR / "TABLE_1_LOCO_headline.csv"
    out.to_csv(out_csv, index=False)
    log.info("Table 1 (LOCO headline) -> %s", out_csv)

    # Table 4: within-cave + baselines.
    if not within_metrics.empty:
        within = within_metrics.copy()
        within["R2"] = within.apply(lambda r: _format_metric_row(r, "r2_mean", "r2_ci_lo", "r2_ci_hi"), axis=1)
        within["RMSE"] = within.apply(lambda r: _format_metric_row(r, "rmse_mean", "rmse_ci_lo", "rmse_ci_hi"), axis=1)
        within["MAE"] = within.apply(lambda r: _format_metric_row(r, "mae_mean", "mae_ci_lo", "mae_ci_hi"), axis=1)
        out2 = within[["target", "model", "n_folds", "R2", "RMSE", "MAE"]].sort_values(["target", "model"])
        out2_csv = METRICS_DIR / "TABLE_4_within_cave_with_baselines.csv"
        out2.to_csv(out2_csv, index=False)
        log.info("Table 4 (within-cave + baselines) -> %s", out2_csv)

    # Save a manifest so the LaTeX side knows what is available.
    manifest = {
        "run_id": RUN_TS,
        "cohort": {
            "n_sites": int(master_table["site_id"].nunique()),
            "n_entities": int(master_table["entity_id"].nunique()),
            "n_samples": int(len(master_table)),
        },
        "models": ALL_MODELS_TABULAR + ALL_MODELS_DL,
        "baselines": BASELINE_NAMES,
        "targets": list(CONFIG.targets),
        "tables": [str(p.relative_to(RUN_DIR))
                   for p in sorted(METRICS_DIR.glob("TABLE_*.csv"))],
        "figures": [str(p.relative_to(RUN_DIR))
                    for p in sorted(FIG_DIR.glob("*.pdf"))],
    }
    (RUN_DIR / "MANIFEST.json").write_text(json.dumps(manifest, indent=2))
    log.info("Run manifest -> %s", RUN_DIR / "MANIFEST.json")
    print(f"\n[OK] Artifacts directory ready: {RUN_DIR}")
    print(f"     Tables : {len(manifest['tables'])} CSVs")
    print(f"     Figures: {len(manifest['figures'])} PDF/PNG pairs")


export_summary_tables()